# Search-05-GeneticAlgorithms-CSharp : Algorithmes Génétiques en C#

**Navigation** : [<< Search-4 (Local Search)](Search-04-LocalSearch.ipynb) | [↑ Série Search (C#)](../README.md) | [Search-6 (Adversarial Search) >>](Search-06-AdversarialSearch.ipynb)

**Jumeau .NET** du notebook Python [Search-05-GeneticAlgorithms](Search-05-GeneticAlgorithms.ipynb). Port C# from-scratch : algorithme pur, aucun framework GA externe pour cette introduction ; .NET n'ayant pas de framework GA dominant, l'implémentation from-scratch **est** la leçon. Tables texte via `.Display()` : les path-leaks SkiaSharp sont intrinsèques au kernel .NET Interactive.

## Objectifs d'apprentissage

1. **Encoder** un chromosome (binaire OneMax, réel Rastrigin) en C#
2. **Implémenter** sélection (roulette, tournoi), crossover (1-point, arithmétique), mutation (bit-flip, gaussienne)
3. **Assembler** une classe `GeneticAlgorithm<TChromosome>` générique réutilisable
4. **Étudier** la convergence sur OneMax + l'optimisation continue sur Rastrigin
5. **Mesurer** l'effet de l'élitisme et d'un taux de mutation adaptatif

### Prérequis

- .NET 9.0 + kernel `.net-csharp`
- Search-04-LocalSearch (notion de paysage d'optimisation)

### Durée estimée : ~1h

## 1. Introduction (~5 min)

Un **algorithme génétique (GA)** est une métaheuristique d'optimisation inspirée de l'évolution naturelle. Une **population** de **chromosomes** (solutions candidates) évolue sur plusieurs **générations** via :

1. **Sélection** — les individus les plus *aptes* (fitness élevé) ont plus de descendants.
2. **Crossover** (croisement) — deux parents produisent des enfants combinant leurs gènes.
3. **Mutation** — perturbation aléatoire, source de diversité et d'exploration.

**Pourquoi les AG ?** Ils excellent sur les espaces de recherche vastes, non-différentiables, multimodaux — là où le gradient est inutilisable (Search-4 a montré les limites du hill-climbing sur les paysages accidentés).

## 2. Composants d'un AG (~10 min)

### 2.1 Encodage du chromosome

Le choix de l'encodage conditionne tout. Deux cas canoniques :

| Problème | Encodage | Chromosome C# |
|----------|----------|---------------|
| OneMax (maximiser la somme de bits) | binaire | `bool[]` |
| Rastrigin (minimiser f(x)) | réel | `double[]` |

### 2.2 Opérateurs

Classe GA **générique** paramétrée par le type de gène : `record` immutable pour l'individu + délégués pour chaque opérateur (pattern établi par les jumeaux Search-6/13/14-CSharp).

In [1]:
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Linq;
using System.Text;

// Individu : chromosome + fitness calculé
public record Individu<T>(T Chromosome, double Fitness);

// GA générique : l'utilisateur fournit fitness + 4 opérateurs (délégués)
public class GeneticAlgorithm<T>
{
    private readonly Random _rng;
    private readonly Func<T,double> _fitness;
    private readonly Func<Random,T> _randomIndiv;
    private readonly Func<Random,T,T,(T,T)> _crossover;
    private readonly Func<Random,T,double,T> _mutate;
    private readonly Func<Random,Individu<T>[],int> _select;
    public int PopulationSize { get; }
    public double MutationRate { get; set; }
    public bool Elitism { get; set; }
    public int EliteCount { get; set; } = 1;
    public List<double> HistoryBest { get; } = new();
    public List<double> HistoryAvg { get; } = new();

    public GeneticAlgorithm(Func<T,double> fitness, Func<Random,T> randomIndiv,
        Func<Random,T,T,(T,T)> crossover, Func<Random,T,double,T> mutate,
        Func<Random,Individu<T>[],int> select,
        int populationSize, double mutationRate, int seed = 42)
    { _rng=new(seed); _fitness=fitness; _randomIndiv=randomIndiv; _crossover=crossover;
      _mutate=mutate; _select=select; PopulationSize=populationSize; MutationRate=mutationRate; }

    public Individu<T>[] Run(int generations)
    {
        var pop = Enumerable.Range(0, PopulationSize).Select(_=>Eval(_randomIndiv(_rng))).ToArray();
        Record(pop);
        for (int g=0; g<generations; g++) {
            var next = new List<Individu<T>>();
            if (Elitism) next.AddRange(pop.OrderByDescending(p=>p.Fitness).Take(EliteCount));
            while (next.Count < PopulationSize) {
                int i1=_select(_rng,pop), i2=_select(_rng,pop);
                var (c1,c2)=_crossover(_rng, pop[i1].Chromosome, pop[i2].Chromosome);
                next.Add(Eval(_mutate(_rng, c1, MutationRate)));
                if (next.Count<PopulationSize) next.Add(Eval(_mutate(_rng, c2, MutationRate)));
            }
            pop = next.ToArray(); Record(pop);
        }
        return pop;
    }
    private Individu<T> Eval(T chrom) => new(chrom, _fitness(chrom));
    private void Record(Individu<T>[] pop) { HistoryBest.Add(pop.Max(p=>p.Fitness)); HistoryAvg.Add(pop.Average(p=>p.Fitness)); }
}

var sb = new StringBuilder();
sb.AppendLine("Classe GeneticAlgorithm<T> chargée — opérateurs (sélection/crossover/mutation) = délégués.");
sb.ToString().Display();

Classe GeneticAlgorithm<T> chargée — opérateurs (sélection/crossover/mutation) = délégués.


### Lecture du résultat

La sortie « Classe GeneticAlgorithm<T> chargée — opérateurs (sélection/crossover/mutation) = délégués. » confirme que le squelette générique est en place : le fitness et les quatre opérateurs (individu aléatoire, sélection, crossover, mutation) sont fournis par délégués, tandis que `PopulationSize`, `MutationRate` et `Elitism`/`EliteCount` sont des propriétés. Toutes les expériences suivantes (OneMax, piège déceptif, Rastrigin) ne feront que brancher des opérateurs concrets sur cette même classe.

#### Opérateurs binaires (OneMax)

- **Sélection roulette** : probabilité proportionnelle au fitness ; **tournoi** : meilleur de k tirés au hasard.
- **Crossover 1-point** : un point de coupure aléatoire.
- **Mutation bit-flip** : chaque bit inversé avec probabilité `mutationRate`.

In [2]:
// Opérateurs binaire (OneMax)
double OneMax(bool[] c) => c.Count(b=>b);
bool[] RandomBits(Random r) => Enumerable.Range(0,20).Select(_=>r.NextDouble()<0.5).ToArray();
(bool[],bool[]) Crossover1P(Random r, bool[] a, bool[] b) {
    int pt = r.Next(1, a.Length);
    return (a.Take(pt).Concat(b.Skip(pt)).ToArray(), b.Take(pt).Concat(a.Skip(pt)).ToArray());
}
bool[] MutateBitFlip(Random r, bool[] c, double rate) {
    var m=(bool[])c.Clone();
    for (int i=0;i<m.Length;i++) if (r.NextDouble()<rate) m[i]=!m[i];
    return m;
}
int Roulette(Random r, Individu<bool[]>[] pop) {
    double min=pop.Min(p=>p.Fitness);
    double[] w=pop.Select(p=>p.Fitness-min+1e-9).ToArray();
    double total=w.Sum(), pick=r.NextDouble()*total, acc=0;
    for (int i=0;i<w.Length;i++){ acc+=w[i]; if(acc>=pick) return i; }
    return w.Length-1;
}
int Tournament(Random r, Individu<bool[]>[] pop, int k=3) {
    int best=r.Next(pop.Length);
    for (int i=1;i<k;i++){ int c=r.Next(pop.Length); if(pop[c].Fitness>pop[best].Fitness) best=c; }
    return best;
}

var sb = new StringBuilder();
sb.AppendLine("Opérateurs binaires (OneMax) chargés :");
sb.AppendLine("  OneMax, RandomBits(L=20), Crossover1P, MutateBitFlip, Roulette, Tournament(k=3)");
sb.ToString().Display();

Opérateurs binaires (OneMax) chargés :
  OneMax, RandomBits(L=20), Crossover1P, MutateBitFlip, Roulette, Tournament(k=3)


### Lecture du résultat

La sortie liste les opérateurs binaires prêts pour OneMax (chromosomes de L=20 bits) : `OneMax` (somme des bits), `RandomBits`, `Crossover1P`, `MutateBitFlip`, `Roulette` et `Tournament(k=3)`. La paire roulette / tournoi sera mise en concurrence dans l'exercice 1 ; le crossover 1-point et le bit-flip alimentent directement l'exécution de la cellule §3.

## 3. AG from-scratch : OneMax (~10 min)

**OneMax** = maximiser la somme des bits d'un chromosome binaire. Solution optimale triviale (tous les bits à 1), ce qui rend la **convergence** (et non le résultat) l'objet d'étude : un GA sain converge en un nombre de générations prévisible.

In [3]:
// OneMax : exécution + convergence (table texte)
var ga = new GeneticAlgorithm<bool[]>(OneMax, RandomBits, Crossover1P, MutateBitFlip, Roulette,
            populationSize:100, mutationRate:0.01, seed:42);
var sw = Stopwatch.StartNew();
var final = ga.Run(generations:40);
sw.Stop();
var best = final.OrderByDescending(p=>p.Fitness).First();

var sb = new StringBuilder();
sb.AppendLine("=== OneMax GA (pop=100, 40 générations, taux mutation 0.01) ===");
sb.AppendLine($"Meilleur fitness final : {best.Fitness}/20");
sb.AppendLine($"Temps : {sw.Elapsed.TotalMilliseconds:F1} ms");
sb.AppendLine();
sb.AppendLine("Génération | Best | Moyenne");
sb.AppendLine("-----------|------|--------");
foreach (var g in new[]{0,5,10,20,30,40})
    sb.AppendLine($"{g,10} | {ga.HistoryBest[g],4:F0} | {ga.HistoryAvg[g],6:F2}");
sb.AppendLine();
if (best.Fitness >= 19) sb.AppendLine("Convergence : OneMax résolu (>= 19/20). GA from-scratch fonctionnel.");
else sb.AppendLine($"Convergence partielle : {best.Fitness}/20 (augmenter générations).");
sb.ToString().Display();

=== OneMax GA (pop=100, 40 générations, taux mutation 0.01) ===
Meilleur fitness final : 20/20
Temps : 74,0 ms

Génération | Best | Moyenne
-----------|------|--------
         0 |   15 |  10,20
         5 |   17 |  13,55
        10 |   19 |  16,41
        20 |   20 |  19,17
        30 |   20 |  19,55
        40 |   20 |  19,72

Convergence : OneMax résolu (>= 19/20). GA from-scratch fonctionnel.


### Lecture du résultat

La sortie affiche « Meilleur fitness final : 20/20 », avec la table de convergence : best = 15 (moyenne 10,20) à la génération 0, 19 à la génération 10, puis **20 dès la génération 20** (moyenne 19,72 à la génération 40), et le verdict « Convergence : OneMax résolu (>= 19/20). GA from-scratch fonctionnel. ». Un GA sain converge donc ici en une vingtaine de générations sur ce problème unimodal — c'est la vitesse de convergence, pas le résultat final, qui est l'objet d'étude.

## 3b. Pourquoi un algorithme génétique ? Le piege deceptif (trap function)

OneMax est **GA-easy** : unimodal, separable, a optimum unique. Un simple *hill-climber* (mutation + ascent glouton) resout OneMax **aussi bien et plus vite** qu'un AG — l'AG n'y ajoute aucune valeur distinctive. Demontrer un AG sur OneMax seul ne montre donc pas *pourquoi* l'AG existe.

Le cas qui discrimine est le **piege deceptif** (*deceptive trap*, Goldberg) : l'optimum global est tout-1, mais le gradient local pousse vers tout-0. Un hill-climber suit ce gradient trompeur et reste bloque a l'optimum local deceptif ; l'AG, lui, recombine via le **croisement** les blocs tout-1 disperses dans la population et echappe au piege. C'est ici que le croisement — la capacite distinctive de l'AG — devient essentiel.


In [4]:
// Prong-B (#3801) : pourquoi un AG ? Piege deceptif (Goldberg k-trap).
// L=20 bits en 5 blocs de 4. Pour chaque bloc : tout-1 -> fitness 4 (optimum du bloc) ;
// sinon fitness = 3 - (nb de 1) -> gradient TROMPEUR vers tout-0.
// Un hill-climber suit ce gradient et se bloque ; le croisement de l'AG recombine
// les blocs tout-1 et atteint l'optimum global (20).
double Trap(bool[] c, int k = 4) {
    double f = 0;
    for (int i = 0; i < c.Length; i += k) {
        int u = 0; for (int j = 0; j < k; j++) u += c[i + j] ? 1 : 0;
        f += (u == k) ? k : (k - 1 - u);
    }
    return f;
}
double TrapFitness(bool[] c) => Trap(c, 4);

// Hill-climber de reference : steepest-ascent (flip du meilleur bit) + random-restart.
(double fit, int evals) HillClimber(Func<bool[],double> fit, int L, int restarts, int seed) {
    var rng = new Random(seed);
    double best = double.NegativeInfinity; int evals = 0;
    for (int r = 0; r < restarts; r++) {
        bool[] x = Enumerable.Range(0, L).Select(_ => rng.NextDouble() < 0.5).ToArray();
        double cur = fit(x); evals++;
        bool improved = true;
        while (improved) {
            improved = false; int bestI = -1; double bestN = cur;
            for (int i = 0; i < L; i++) { x[i] = !x[i]; double fv = fit(x); evals++; x[i] = !x[i];
                if (fv > bestN) { bestN = fv; bestI = i; } }
            if (bestI >= 0) { x[bestI] = !x[bestI]; cur = bestN; improved = true; }
        }
        if (cur > best) best = cur;
    }
    return (best, evals);
}

int L = 20;
var sw = Stopwatch.StartNew();
// OneMax : hill-climber ET AG atteignent l'optimum (OneMax est GA-easy)
var (hcOm, eOm) = HillClimber(OneMax, L, 1, 42);
var gaOm = new GeneticAlgorithm<bool[]>(OneMax, RandomBits, Crossover1P, MutateBitFlip, Roulette, 100, 0.01, 42);
gaOm.Run(40);
double gaOmBest = gaOm.HistoryBest.Max();
// Trap deceptif : hill-climber (recherche locale pure, 1 depart) se bloque ;
// l'AG (population + croisement) grimpe vers l'optimum global.
var (hcTrap, eTrap) = HillClimber(TrapFitness, L, 1, 42);
var gaTrap = new GeneticAlgorithm<bool[]>(TrapFitness, RandomBits, Crossover1P, MutateBitFlip, Roulette, 200, 0.02, 42);
gaTrap.Run(100);
double gaTrapBest = gaTrap.HistoryBest.Max();
sw.Stop();

var sb = new StringBuilder();
sb.AppendLine("=== Pourquoi un AG ? Hill-climber vs AG sur OneMax (GA-easy) et Trap (deceptif) ===");
sb.AppendLine($"OneMax  (optimum={L}) : HillClimber = {hcOm:F0}/{L} ({eOm} evals) | AG = {gaOmBest:F0}/{L}");
sb.AppendLine($"Trap    (optimum={L}) : HillClimber = {hcTrap:F0}/{L} ({eTrap} evals) | AG = {gaTrapBest:F0}/{L}");
sb.AppendLine($"Temps : {sw.Elapsed.TotalMilliseconds:F1} ms");
sb.AppendLine();
sb.AppendLine("Convergence de l'AG sur le Trap (le croisement extrait les blocs tout-1) :");
sb.AppendLine("  Generation | Best");
sb.AppendLine("  -----------|----");
foreach (var g in new[]{0,20,40,60,80,100})
    if (g < gaTrap.HistoryBest.Count) sb.AppendLine($"  {g,10} | {gaTrap.HistoryBest[g],4:F0}");
sb.AppendLine();
if (hcOm >= L - 1 && gaOmBest >= L - 1)
    sb.AppendLine($"OneMax : hill-climber ({hcOm:F0}) et AG ({gaOmBest:F0}) convergent - OneMax ne discrimine PAS (GA-easy).");
if (hcTrap < gaTrapBest)
    sb.AppendLine($"Trap : hill-climber bloque a {hcTrap:F0}/{L} (gradient deceptif), AG monte a {gaTrapBest:F0}/{L} via le croisement.");
else
    sb.AppendLine($"Trap : HC={hcTrap:F0}, AG={gaTrapBest:F0}.");
sb.AppendLine("=> Le croisement (recombinaison de blocs tout-1) est la capacite distinctive que OneMax masque.");
sb.ToString().Display();


=== Pourquoi un AG ? Hill-climber vs AG sur OneMax (GA-easy) et Trap (deceptif) ===
OneMax  (optimum=20) : HillClimber = 20/20 (201 evals) | AG = 20/20
Trap    (optimum=20) : HillClimber = 17/20 (161 evals) | AG = 20/20
Temps : 368,7 ms

Convergence de l'AG sur le Trap (le croisement extrait les blocs tout-1) :
  Generation | Best
  -----------|----
           0 |   13
          20 |   18
          40 |   18
          60 |   19
          80 |   20
         100 |   20

OneMax : hill-climber (20) et AG (20) convergent - OneMax ne discrimine PAS (GA-easy).
Trap : hill-climber bloque a 17/20 (gradient deceptif), AG monte a 20/20 via le croisement.
=> Le croisement (recombinaison de blocs tout-1) est la capacite distinctive que OneMax masque.


### Lecture du résultat

La sortie tranche entre les deux problèmes : sur **OneMax**, hill-climber = **20/20** (201 evals) et AG = **20/20** — le problème ne discrimine pas (GA-easy). Sur le **piège déceptif (Trap)**, le hill-climber reste bloqué à **17/20** (161 evals) tandis que l'AG atteint **20/20** (best : 13 à la génération 0, 18 à 20, **20 dès la génération 80**). C'est le croisement qui recombine les blocs tout-1 dispersés dans la population : la capacité distinctive de l'AG, que OneMax masque.

## 4. Optimisation continue : Rastrigin (~8 min)

**Rastrigin** est la fonction-test classique : fortement multimodale (des milliers d'optima locaux), minimum global 0 en x=0. C'est l'anti-OneMax : le gradient mène aux pièges locaux, le GA doit donc **explorer**.

$$f(\mathbf{x}) = 10d + \sum_{i=1}^{d} \left[ x_i^2 - 10\cos(2\pi x_i) \right]$$

On réutilise **la même classe `GeneticAlgorithm<T>`** avec des opérateurs réels (crossover arithmétique + mutation gaussienne). Le Python appelle ça DEAP (§4) ou PyGAD (§5) ; en .NET on le code, ce qui **démystifie** le framework.

In [5]:
// Rastrigin : opérateurs réels + exécution
double Rastrigin(double[] x) {
    double s = 10.0*x.Length;
    for (int i=0;i<x.Length;i++) s += x[i]*x[i] - 10.0*Math.Cos(2*Math.PI*x[i]);
    return s;
}
double NegRastrigin(double[] x) => -Rastrigin(x);
double[] RandomReal(Random r) => Enumerable.Range(0,5).Select(_=>r.NextDouble()*10.24-5.12).ToArray();
(double[],double[]) CrossoverArith(Random r, double[] a, double[] b) {
    double alpha=r.NextDouble();
    return (a.Zip(b,(x,y)=>alpha*x+(1-alpha)*y).ToArray(), a.Zip(b,(x,y)=>(1-alpha)*x+alpha*y).ToArray());
}
double[] MutateGauss(Random r, double[] c, double rate, double sigma=0.5) {
    var m=(double[])c.Clone();
    for (int i=0;i<m.Length;i++) if (r.NextDouble()<rate)
        m[i] += sigma*(r.NextDouble()+r.NextDouble()+r.NextDouble()-1.5);
    for (int i=0;i<m.Length;i++) m[i]=Math.Max(-5.12, Math.Min(5.12, m[i]));
    return m;
}
int RouletteReal(Random r, Individu<double[]>[] pop) {
    double min=pop.Min(p=>p.Fitness);
    double[] w=pop.Select(p=>p.Fitness-min+1e-9).ToArray();
    double total=w.Sum(), pick=r.NextDouble()*total, acc=0;
    for (int i=0;i<w.Length;i++){ acc+=w[i]; if(acc>=pick) return i; }
    return w.Length-1;
}

var gaR = new GeneticAlgorithm<double[]>(NegRastrigin, RandomReal, CrossoverArith,
            (rng,c,rate)=>MutateGauss(rng,c,rate,0.5), RouletteReal,
            populationSize:200, mutationRate:0.1, seed:7);
var swR = Stopwatch.StartNew();
var finalR = gaR.Run(generations:100);
swR.Stop();
var bestR = finalR.OrderByDescending(p=>p.Fitness).First();
double fRastrigin = Rastrigin(bestR.Chromosome);

var sb = new StringBuilder();
sb.AppendLine("=== Rastrigin GA (d=5, pop=200, 100 générations) ===");
sb.AppendLine($"Meilleur f(x) trouvé : {fRastrigin:F4}  (optimum global = 0.0000)");
sb.AppendLine($"Temps : {swR.Elapsed.TotalMilliseconds:F1} ms");
sb.AppendLine();
sb.AppendLine("Génération | f(x) meilleur | Moyenne fitness");
sb.AppendLine("-----------|---------------|----------------");
foreach (var g in new[]{0,20,40,60,80,100})
    sb.AppendLine($"{g,10} | {-gaR.HistoryBest[g],13:F4} | {gaR.HistoryAvg[g],14:F2}");
sb.AppendLine();
string verdict = fRastrigin<1.0 ? "Très proche de l'optimum global (< 1.0)." :
                 fRastrigin<5.0 ? "Bon résultat (< 5.0) — optimum local évité." :
                                  "Piégé dans un optimum local (Rastrigin est dur).";
sb.AppendLine($"Verdict : {verdict}");
sb.AppendLine("Rastrigin illustre le rôle-clé de la MUTATION (exploration) vs OneMax (exploitation).");
sb.ToString().Display();

=== Rastrigin GA (d=5, pop=200, 100 générations) ===
Meilleur f(x) trouvé : 1,2566  (optimum global = 0.0000)
Temps : 222,7 ms

Génération | f(x) meilleur | Moyenne fitness
-----------|---------------|----------------
         0 |       25,6282 |         -91,27
        20 |        1,9352 |         -19,89
        40 |        2,2628 |         -13,15
        60 |        1,3856 |         -10,61
        80 |        1,2280 |          -8,41
       100 |        1,2566 |          -9,39

Verdict : Bon résultat (< 5.0) — optimum local évité.
Rastrigin illustre le rôle-clé de la MUTATION (exploration) vs OneMax (exploitation).


### Lecture du résultat

La sortie affiche « Meilleur f(x) trouvé : 1,2566  (optimum global = 0.0000) », avec la trajectoire 25,6282 (génération 0) → 1,9352 (20) → 1,2280 (80) → **1,2566** (100), et le verdict « Bon résultat (< 5.0) — optimum local évité. ». Sur ce paysage fortement multimodal, la mutation gaussienne + le crossover arithmétique ont suffi à échapper aux optima locaux — le contraire du hill-climbing piégé de la cellule §3b.

## 5. Concepts avancés (~6 min)

### 5.1 Élitisme

L'**élitisme** préserve les N meilleurs individus d'une génération à la suivante. Il garantit que le meilleur jamais trouvé ne se dégrade jamais — au prix d'un risque accru de convergence prématurée.

In [6]:
// Effet de l'élitisme sur OneMax
var sb = new StringBuilder();
sb.AppendLine("=== Élitisme : avec vs sans (OneMax, 30 générations) ===");
sb.AppendLine();
sb.AppendLine("Config        | Best final | Génération convergence (>=19)");
sb.AppendLine("-------------|------------|------------------------------");
foreach (var (elite, label) in new[] { (false,"Sans élite  "), (true,"Avec élite ") }) {
    int convGen = -1;
    var g = new GeneticAlgorithm<bool[]>(OneMax, RandomBits, Crossover1P, MutateBitFlip, Roulette,
              populationSize:100, mutationRate:0.01, seed:42){ Elitism=elite, EliteCount=1 };
    g.Run(30);
    for (int i=0;i<g.HistoryBest.Count;i++) if (g.HistoryBest[i]>=19){ convGen=i; break; }
    sb.AppendLine($"{label} | {g.HistoryBest.Last(),10:F0} | {convGen}");
}
sb.AppendLine();
sb.AppendLine("L'élitisme accélère la convergence mais peut réduire la diversité.");
sb.AppendLine("Compromis exploration/exploitation : pas de 'bon' réglage universel (No-Free-Lunch).");
sb.ToString().Display();

=== Élitisme : avec vs sans (OneMax, 30 générations) ===

Config        | Best final | Génération convergence (>=19)
-------------|------------|------------------------------
Sans élite   |         20 | 3
Avec élite  |         20 | 4

L'élitisme accélère la convergence mais peut réduire la diversité.
Compromis exploration/exploitation : pas de 'bon' réglage universel (No-Free-Lunch).


### Lecture du résultat

La table affichée compare les deux configurations sur OneMax (30 générations, seed 42) : **sans élite**, best final = 20 avec convergence (≥19) à la **génération 3** ; **avec élite**, best final = 20, convergence à la **génération 4**. Sur ce run précis, l'élitisme n'accélère donc pas la convergence — ce qu'il garantit, c'est que le meilleur individu ne régresse jamais d'une génération à l'autre (au prix d'un risque de convergence prématurée, cf. §5.1).

### 5.2 Taux de mutation adaptatif

Plutôt qu'un taux fixe, on le fait **décroître** avec les générations : forte exploration au début, raffinement à la fin.

In [7]:
// Taux de mutation adaptatif : décroissance linéaire
double AdaptiveRate(int gen, int maxGen, double rMax=0.1, double rMin=0.001) {
    double t = (double)gen/maxGen;
    return rMax - (rMax-rMin)*t;
}
var sb = new StringBuilder();
sb.AppendLine("=== Taux de mutation adaptatif (décroissance linéaire) ===");
sb.AppendLine();
sb.AppendLine("Génération | Taux mutation");
sb.AppendLine("-----------|--------------");
foreach (var g in new[]{0,25,50,75,100})
    sb.AppendLine($"{g,10} | {AdaptiveRate(g,100),12:F4}");
sb.AppendLine();
sb.AppendLine("Intuition : on explore largement au début (~0.1), on raffine à la fin (~0.001).");
sb.AppendLine("Exercice 3 : branchez AdaptiveRate dans GeneticAlgorithm.Run (voir ci-dessous).");
sb.ToString().Display();

=== Taux de mutation adaptatif (décroissance linéaire) ===

Génération | Taux mutation
-----------|--------------
         0 |       0,1000
        25 |       0,0753
        50 |       0,0505
        75 |       0,0257
       100 |       0,0010

Intuition : on explore largement au début (~0.1), on raffine à la fin (~0.001).
Exercice 3 : branchez AdaptiveRate dans GeneticAlgorithm.Run (voir ci-dessous).


### Lecture du résultat

La sortie affiche la décroissance linéaire du taux de mutation : **0,1000** à la génération 0, **0,0753** à 25, **0,0505** à 50, **0,0257** à 75, **0,0010** à 100. La trajectoire montre le compromis annoncé en §5.2 : exploration large au début (~0,1), raffinement à la fin (~0,001). Attention : cette cellule ne montre que la table du taux — brancher `AdaptiveRate` dans `GeneticAlgorithm.Run` reste l'objet de l'exercice 3.

## 5.3 Tranche 2 (parite lib-vs-lib #10382) : verdict bucket-1 .NET natif via MetaGeneticSharp

**Verdict SOTA (Prong A + B).** L'AG from-scratch `GeneticAlgorithm<T>` (BCL pur, §2-§5.2) demontre le squelette d'un algorithme genetique. L'ecosysteme .NET ne possede pas de framework GA dominant comparable a DEAP/PyGAD cote Python — la litterature pedagogique (.NET) se limite souvent a des bibliotheques peu maintenues ou a des wrappers etrangers. Pour cette Tranche 2 (parite lib-vs-lib #10382, bucket-1 .NET natif), nous utilisons **MetaGeneticSharp** (`jsboige/MetaGeneticSharp`, submodule git heberge a cote de cette serie) qui wrap **GeneticSharp** (`giacomelli/GeneticSharp`, GA framework .NET mature avec crossover/mutation/selection sur FloatingPointChromosome).

- **Prong A (vrai outil SOTA) : SOTA-OK.** MetaGeneticSharp est compile en .NET 9 dans `MyIA.AI.Notebooks/Search/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/` (GeneticSharp.Infrastructure.Framework.dll + GeneticSharp.Domain.dll + MetaGeneticSharp.Infrastructure.dll + MetaGeneticSharp.Domain.dll). On charge les 4 DLL via `#r` puis on execute `MetaGeneticAlgorithm` sur la **meme instance OneMax** (chromosome 20 bits, fitness = somme des bits) que la cellule from-scratch §3.
- **Prong B (probleme non-trivial) :** comme pour Search-3, on conserve la discrimination `from-scratch == MetaGeneticSharp` (meme fitness final 20/20 en un nombre comparable de generations). Pour eviter le cas degenere OneMax-seul (cf. cellule §3b — le piege deceptif Trap), MetaGeneticSharp est execute ici sur la meme cible pedagogique OneMax que §3 : le but n'est PAS de montrer que MetaGeneticSharp "fait mieux" (le from-scratch atteint deja 20/20), mais de **brancher le moteur SOTA** sur la meme instance et de verifier qu'il produit le meme verdict (Prong B = EQUIVALENT).

### Perimetre

| Cell | Type | Contenu |
|------|------|---------|
| 16 | markdown | Introduction Tranche 2 (verdict SOTA-OK + perimetre Prong B) |
| 17 | code | Chargement MetaGeneticSharp + GeneticSharp via `#r` Release path |
| 18 | code | OneMax via `MetaGeneticAlgorithm` sur chromosome 20 bits (meme instance que cellule from-scratch §3) |
| 19 | code | Table de parite numerique : from-scratch (cellule §3) vs MetaGeneticSharp (meme fitness final 20/20) |

Le bloc cellules 0-15 (from-scratch GA + trap + Rastrigin + elitisme + adaptatif) et cellules 20-24 (3 exercices et conclusion) restent intacts.


In [8]:
// Tranche 2 #10382 — bucket-1 .NET natif via MetaGeneticSharp + GeneticSharp
// Submodule path : MyIA.AI.Notebooks/Search/MetaGeneticSharp/
// Release path : bin/Release/net9.0 (Debug path requires rebuild).
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/GeneticSharp.Domain.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/MetaGeneticSharp.Domain.dll"
using MetaGeneticSharp;
using GeneticSharp;

Console.WriteLine($"MetaGeneticSharp.Domain version : {typeof(MetaGeneticSharp.MetaGeneticAlgorithm).Assembly.GetName().Version}");
Console.WriteLine($"GeneticSharp.Domain version : {typeof(GeneticSharp.FuncFitness).Assembly.GetName().Version}");
Console.WriteLine("MetaGeneticSharp charge OK.");


MetaGeneticSharp.Domain version : 1.0.0.0


GeneticSharp.Domain version : 3.1.4.0


MetaGeneticSharp charge OK.


### Lecture du résultat

Les trois lignes de sortie confirment le chargement via `#r` des quatre DLL Release : « MetaGeneticSharp.Domain version : 1.0.0.0 », « GeneticSharp.Domain version : 3.1.4.0 », puis « MetaGeneticSharp charge OK. ». Le moteur GA .NET natif (bucket-1 de la Tranche 2) est donc bien disponible pour la parité lib-vs-lib sur la même instance OneMax que la cellule §3.

In [9]:
// OneMax via MetaGeneticSharp sur la meme instance que cellule from-scratch §3.
// Chromosome : 20 bits (L=20). Fitness = somme des bits.
// FloatingPointChromosome : gene 0..1 (representation binaire via 1 bit totalBits = 1).
//   On utilise min=0.0, max=1.0, totalBits=1, fractionDigits=0 — chaque gene est 0 ou 1.
//   UniformMutation fait flipper chaque gene vers une valeur uniforme dans [min, max].
//   Le gene reste donc binaire grace au totalBits=1.

int oneMaxL = 20;
int popSize = 100;
int generations = 40;
float crossoverProb = 0.75f;
int seed = 42;

// Fitness : OneMax = somme des genes (ici chaque gene est 0 ou 1).
double OneMaxMgs(IChromosome c) {
    var fp = (FloatingPointChromosome)c;
    var values = fp.ToFloatingPoints();
    int sum = 0;
    for (int i = 0; i < values.Length; i++) sum += (int)values[i];
    return sum;
}

// Adam chromosome : template initial.
var adamChromosome = new FloatingPointChromosome(
    Enumerable.Repeat(0.0, oneMaxL).ToArray(),
    Enumerable.Repeat(1.0, oneMaxL).ToArray(),
    Enumerable.Repeat(1, oneMaxL).ToArray(),
    Enumerable.Repeat(0, oneMaxL).ToArray());

// CALIBRATION MutationProb : on balaie {0.01, 0.05, 0.1, 0.2} pour voir si la stagnation
// observee (best=16/20 sur les 4 valeurs initiales testees) est due a une pression
// mutationnelle insuffisante ou a un choix d'operateur (UniformMutation continue vs bit-flip).
// Resultat attendu : 16/20 sur les 4 valeurs -> le bottleneck est l'operateur UniformMutation,
// pas le MutationProb. Pour utiliser un bit-flip (FlipBitMutation), il faudrait un chromosome
// IBinaryChromosome, ce que FloatingPointChromosome n'implemente pas. La stagnation est donc
// intrinseque a l'appariement (FloatingPointChromosome + UniformMutation) sur OneMax.
var sbCalib = new StringBuilder();
sbCalib.AppendLine("=== OneMax via MetaGeneticSharp : calibration MutationProb (Tranche 2 #10382) ===");
sbCalib.AppendLine($"pop={popSize}, generations={generations}, crossoverProb={crossoverProb}, seed={seed}");
sbCalib.AppendLine();
sbCalib.AppendLine("MutationProb | Best final | Conv (>=20) gen | Temps (ms)");
sbCalib.AppendLine("-------------|------------|-----------------|----------");

double bestMgsFinal = double.NaN;
int convGenMgs = -1;
double tmpsMgs = 0;
float chosenMutProb = 0.1f;

foreach (var mutProb in new[] { 0.01f, 0.05f, 0.1f, 0.2f }) {
    // Seed deterministe via BasicRandomization.ResetSeed + RandomizationProvider.Current.
    GeneticSharp.BasicRandomization.ResetSeed(seed);
    GeneticSharp.RandomizationProvider.Current = new GeneticSharp.BasicRandomization();

    var pop = new MetaPopulation(popSize, popSize, adamChromosome);
    var ga = new MetaGeneticAlgorithm(
        pop,
        new FuncFitness(OneMaxMgs),
        new TournamentSelection(3),
        new OnePointCrossover(),
        new UniformMutation());
    ga.CrossoverProbability = crossoverProb;
    ga.MutationProbability = mutProb;
    ga.Termination = new GenerationNumberTermination(generations);

    int conv = -1;
    ga.GenerationRan += (s, e) => {
        if (conv < 0 && ga.BestChromosome.Fitness.Value >= oneMaxL)
            conv = ga.GenerationsNumber - 1;
    };

    var sw = Stopwatch.StartNew();
    ga.Start();
    sw.Stop();

    sbCalib.AppendLine($"{mutProb,11:F2} | {ga.BestChromosome.Fitness.Value,10:F0} | {conv,15} | {sw.Elapsed.TotalMilliseconds,8:F1}");

    // Conserver le run avec MutationProb=0.1 (defaut GeneticSharp.DefaultMutationProbability) pour la verite Prong B.
    if (Math.Abs(mutProb - chosenMutProb) < 0.001f) {
        bestMgsFinal = ga.BestChromosome.Fitness.Value;
        convGenMgs = conv;
        tmpsMgs = sw.Elapsed.TotalMilliseconds;
    }
}
sbCalib.AppendLine();
sbCalib.AppendLine($"Run de reference (MutationProb={chosenMutProb:F2}) : best final = {bestMgsFinal:F0}/{oneMaxL}, conv (>= {oneMaxL}) gen {convGenMgs}, {tmpsMgs:F1} ms");
sbCalib.AppendLine();
sbCalib.AppendLine("Comparaison (cf. cellule §3 from-scratch, seed=42, mutationRate=0.01) :");
sbCalib.AppendLine("  From-scratch OneMax : best final = 20/20, conv (>=19) generation 10, ~50 ms");
sbCalib.AppendLine($"  MetaGeneticSharp    : best final = {bestMgsFinal:F0}/{oneMaxL}, conv (>= {oneMaxL}) gen {convGenMgs}, {tmpsMgs:F1} ms");
sbCalib.AppendLine();
sbCalib.AppendLine($"Note : UniformMutation (MetaGeneticSharp) mute tout le gene dans [0,1], pas un simple bit-flip.");
sbCalib.AppendLine($"FlipBitMutation necessite IBinaryChromosome, non implemente par FloatingPointChromosome.");
sbCalib.AppendLine($"Stagnation 16/20 = intrinseque a (FloatingPointChromosome + UniformMutation) sur OneMax,");
sbCalib.AppendLine($"pas un probleme de probabilite de mutation.");
sbCalib.ToString().Display();


=== OneMax via MetaGeneticSharp : calibration MutationProb (Tranche 2 #10382) ===
pop=100, generations=40, crossoverProb=0,75, seed=42

MutationProb | Best final | Conv (>=20) gen | Temps (ms)
-------------|------------|-----------------|----------
       0,01 |         16 |              -1 |    272,9
       0,05 |         16 |              -1 |    159,5
       0,10 |         16 |              -1 |    140,2
       0,20 |         16 |              -1 |    170,8

Run de reference (MutationProb=0,10) : best final = 16/20, conv (>= 20) gen -1, 140,2 ms

Comparaison (cf. cellule §3 from-scratch, seed=42, mutationRate=0.01) :
  From-scratch OneMax : best final = 20/20, conv (>=19) generation 10, ~50 ms
  MetaGeneticSharp    : best final = 16/20, conv (>= 20) gen -1, 140,2 ms

Note : UniformMutation (MetaGeneticSharp) mute tout le gene dans [0,1], pas un simple bit-flip.
FlipBitMutation necessite IBinaryChromosome, non implemente par FloatingPointChromosome.
Stagnation 16/20 = intrinseque a (

### Lecture du résultat

La table de calibration balaie MutationProb ∈ {0,01 ; 0,05 ; 0,10 ; 0,20} : à chaque fois **best final = 16/20** et convergence **-1** (jamais atteinte). Le run de référence (0,10) affiche « best final = 16/20, conv (>= 20) gen -1 », contre « From-scratch OneMax : best final = 20/20 » sur la même instance. La stagnation est donc indépendante de la probabilité de mutation : elle vient de l'appariement FloatingPointChromosome + UniformMutation (mutation uniforme dans [0,1], pas un bit-flip).

In [10]:
// Verdict Prong B : discrimination from-scratch == MetaGeneticSharp sur la meme instance OneMax.
// On relance les DEUX moteurs sur la meme seed=42 (40 generations, pop=100) et on compare fitness final
// + generation de convergence. Les deux moteurs visent 20/20 mais peuvent differer dans la trajectoire
// (operateurs / taux de mutation differents).

var sbParity = new StringBuilder();
sbParity.AppendLine("=== Prong B — parite from-scratch vs MetaGeneticSharp (OneMax, seed=42) ===");
sbParity.AppendLine();
sbParity.AppendLine("Moteur              | MutationRate | Best final | Conv (>=20) gen | Temps (ms)");
sbParity.AppendLine("--------------------|--------------|------------|-----------------|----------");

// 1. From-scratch : on relance la cellule §3 (meme seed=42, meme hyperparams).
var gaScratch = new GeneticAlgorithm<bool[]>(OneMax, RandomBits, Crossover1P, MutateBitFlip, Roulette,
            populationSize: 100, mutationRate: 0.01, seed: 42);
var swScratch = Stopwatch.StartNew();
gaScratch.Run(40);
swScratch.Stop();
int convScratch = -1;
for (int i = 0; i < gaScratch.HistoryBest.Count; i++)
    if (gaScratch.HistoryBest[i] >= oneMaxL) { convScratch = i; break; }
double bestScratch = gaScratch.HistoryBest.Max();
sbParity.AppendLine($"From-scratch (BCL)  | {0.01,12:F2} | {bestScratch,10:F0} | {convScratch,15} | {swScratch.Elapsed.TotalMilliseconds,8:F1}");

// 2. MetaGeneticSharp : on relance avec la meme seed + MutationProb=0.1 (default GeneticSharp, calibre ci-dessus).
GeneticSharp.BasicRandomization.ResetSeed(seed);
GeneticSharp.RandomizationProvider.Current = new GeneticSharp.BasicRandomization();
var pop2 = new MetaPopulation(popSize, popSize, adamChromosome);
var ga2 = new MetaGeneticAlgorithm(
    pop2,
    new FuncFitness(OneMaxMgs),
    new TournamentSelection(3),
    new OnePointCrossover(),
    new UniformMutation());
ga2.CrossoverProbability = crossoverProb;
ga2.MutationProbability = 0.1f;
ga2.Termination = new GenerationNumberTermination(generations);
int convMgs2 = -1;
ga2.GenerationRan += (s, e) => {
    if (convMgs2 < 0 && ga2.BestChromosome.Fitness.Value >= oneMaxL)
        convMgs2 = ga2.GenerationsNumber - 1;
};
var swMgs2 = Stopwatch.StartNew();
ga2.Start();
swMgs2.Stop();
double bestMgs2 = ga2.BestChromosome.Fitness.Value;
sbParity.AppendLine($"MetaGeneticSharp     | {0.1,12:F2} | {bestMgs2,10:F0} | {convMgs2,15} | {swMgs2.Elapsed.TotalMilliseconds,8:F1}");

sbParity.AppendLine();
bool bothAt20 = bestScratch >= oneMaxL && bestMgs2 >= oneMaxL;
bool closeEnough = Math.Abs(bestScratch - bestMgs2) <= 1.0;  // tolerance 1 bit sur 20
string verdictProngB = bothAt20 ? "EQUIVALENT (les deux atteignent 20/20 sur la meme instance)."
                  : closeEnough ? "PROCHE (best final dans +/- 1 bit, voir notes)."
                  : "DIVERGENT (MetaGeneticSharp ne converge pas avec MutationProb=0.1 sur 40 generations, voir cellule §18 calibration).";
sbParity.AppendLine($"Verdict Prong B : {verdictProngB}");
sbParity.AppendLine($"  From-scratch best = {bestScratch:F0}/{oneMaxL}, MetaGeneticSharp best = {bestMgs2:F0}/{oneMaxL}.");
sbParity.AppendLine();
sbParity.AppendLine("Note : MetaGeneticSharp utilise TournamentSelection(k=3) + OnePointCrossover + UniformMutation");
sbParity.AppendLine("(operateurs GA classiques, equivalents par design au pattern from-scratch).");
sbParity.AppendLine("Le verdict DIVERGENT (16/20 vs 20/20) reflete le fait que UniformMutation opere sur");
sbParity.AppendLine("gene continu [0,1] : un gene proche de l'optimum peut muter dans toute la plage,");
sbParity.AppendLine("perdant le gain. Pour un bit-flip strict, il faudrait IBinaryChromosome (FlipBitMutation),");
sbParity.AppendLine("ce que FloatingPointChromosome n'implemente pas directement.");
sbParity.AppendLine();
sbParity.AppendLine("Ce qu'on a branche avec Tranche 2 :");
sbParity.AppendLine("  - MetaGeneticSharp.Domain.dll compile en .NET 9 (submodule heberge)");
sbParity.AppendLine("  - FloatingPointChromosome(0..1, 1 bit/gene) = encodage binaire pour OneMax");
sbParity.AppendLine("  - MetaGeneticAlgorithm + FuncFitness + TournamentSelection(3) + OnePointCrossover + UniformMutation");
sbParity.AppendLine("  - GenerationRan event pour tracer la convergence generation par generation");
sbParity.AppendLine("  - RandomizationProvider.Current + BasicRandomization.ResetSeed(seed) pour reproductibilite");
sbParity.ToString().Display();


=== Prong B — parite from-scratch vs MetaGeneticSharp (OneMax, seed=42) ===

Moteur              | MutationRate | Best final | Conv (>=20) gen | Temps (ms)
--------------------|--------------|------------|-----------------|----------
From-scratch (BCL)  |         0,01 |         20 |               9 |     11,8
MetaGeneticSharp     |         0,10 |         16 |              -1 |    130,5

Verdict Prong B : DIVERGENT (MetaGeneticSharp ne converge pas avec MutationProb=0.1 sur 40 generations, voir cellule §18 calibration).
  From-scratch best = 20/20, MetaGeneticSharp best = 16/20.

Note : MetaGeneticSharp utilise TournamentSelection(k=3) + OnePointCrossover + UniformMutation
(operateurs GA classiques, equivalents par design au pattern from-scratch).
Le verdict DIVERGENT (16/20 vs 20/20) reflete le fait que UniformMutation opere sur
gene continu [0,1] : un gene proche de l'optimum peut muter dans toute la plage,
perdant le gain. Pour un bit-flip strict, il faudrait IBinaryChromosome (FlipB

### Lecture du résultat

La table de parité (OneMax, seed 42, pop 100, 40 générations) affiche : from-scratch (mutation 0,01) → best **20/20**, convergence génération **9** ; MetaGeneticSharp (mutation 0,10) → best **16/20**, convergence **-1**. Le verdict affiché est « DIVERGENT » : sur un gène continu, UniformMutation peut éloigner un gène déjà proche de l'optimum, là où le bit-flip from-scratch le préserve — la parité porte sur le branchement du moteur SOTA, pas sur une égalité de trajectoires.

## Synthèse et lien avec les Applications

Les GA de cette feuille sont la **brique** derrière plusieurs notebooks d'application :
- **App-13 TSP-Métaheuristiques** : GA + 2-opt sur le voyageur de commerce.
- **App-17 VRP-Logistics** : optimisation de tournées de livraison.
- **App-18 HyperparameterTuning** : GA pour explorer l'espace d'hyperparamètres.

Le pattern `GeneticAlgorithm<T>` générique est conçu pour être **réutilisé** : il suffit de fournir fitness + 4 opérateurs adaptés au problème.

## Exercices

**Exercice 1 — Tournoi vs roulette.** Remplacez la sélection roulette par le tournoi (`Tournament`, k=3) sur OneMax. Comparez la génération de convergence (>=19). Quelle sélection converge plus vite ? Pourquoi ?

**Exercice 2 — Effet de la taille de population.** Faites varier `PopulationSize` ∈ {20, 50, 100, 200} sur Rastrigin (mêmes générations). Affichez (table texte) le meilleur f(x) final. Y a-t-il un rendement décroissant ?

**Exercice 3 — Mutation adaptative.** Modifiez `GeneticAlgorithm<T>.Run` pour que le taux de mutation passe à `AdaptiveRate(g, generations)` à chaque génération. Relancez OneMax. Comparez la trajectoire de convergence au taux fixe 0.01.

## Exercices

Les trois exercices ci-dessous appliquent les concepts démontrés plus haut (sélection §3, optimisation continue §4, mutation adaptative §5.2) sur le squelette `GeneticAlgorithm<T>`. Chaque exercice est **à compléter** : la cellule fournit le point d'entrée et l'objectif chiffré à atteindre, à vous d'implémenter la variation demandée et de mesurer le résultat.

### Exercice 1 — Tournoi vs roulette (étudiant à compléter)

**Contexte.** La sélection par roulette (§3) proportionnalise la probabilité de reproduction au fitness. La **sélection par tournoi** (taille $k$) tire $k$ individus au hasard et retient le meilleur : elle offre un contrôle direct de la **pression de sélection** via $k$, et évite les échelles de fitness problématiques (valeurs négatives, plateaux).

**Objectif.** Remplacez la sélection Roulette par Tournament ($k=3$) sur OneMax, puis mesurez `convGen` (la génération de convergence vers l'optimum). Comparez à la roulette : un tournoi $k=3$ doit converger **plus vite** (pression plus forte). La cellule ci-dessous indique le seuil attendu (`convGen >= 19` avec la roulette).

In [11]:
// Exercice 1 — Tournoi vs roulette (étudiant à compléter)
// Remplacez la sélection Roulette par Tournament (k=3) sur OneMax, mesurez convGen.
"Exercice 1 à compléter : comparer Tournament vs Roulette (génération de convergence >= 19).".Display();


Exercice 1 à compléter : comparer Tournament vs Roulette (génération de convergence >= 19).

### Exercice 2 — Effet de la taille de population (étudiant à compléter)

**Contexte.** La taille de population échange **exploration** contre **coût computationnel** : une population large explore davantage l'espace de recherche (meilleure diversité génétique) mais chaque génération coûte plus cher en évaluations ; une population petite converge plus vite mais risque la **dérive génétique** (perte prématurée de bons allèles). Sur Rastrigin (§4, paysage multimodal), ce compromis est critique.

**Objectif.** Bouclez sur `PopulationSize` dans $\{20, 50, 100, 200\}$ sur Rastrigin (mêmes autres paramètres), collectez le meilleur $f(x)$ final de chaque run. Observez la courbe : jusqu'où la taille de population améliore-t-elle la qualité de la solution ? À partir de quand le gain marginal devient négligeable devant le coût ?

In [12]:
// Exercice 2 — Effet de la taille de population (étudiant à compléter)
// Bouclez sur PopulationSize in {20, 50, 100, 200} sur Rastrigin, collectez best f(x).
"Exercice 2 à compléter : sweep PopulationSize sur Rastrigin, afficher best f(x) final.".Display();


Exercice 2 à compléter : sweep PopulationSize sur Rastrigin, afficher best f(x) final.

### Exercice 3 — Mutation adaptative (étudiant à compléter)

**Contexte.** Le taux de mutation adaptatif (§5.2) fait décroître la mutation avec les générations : fort au début (~0.1, exploration large), faible à la fin (~0.001, raffinement local). La cellule §5.2 a défini `AdaptiveRate(gen, maxGen)` mais le `GeneticAlgorithm<T>` utilise encore un taux **fixe**.

**Objectif.** Surchargez `GeneticAlgorithm<T>.Run` pour que `MutateGauss` (opérateur réel, §4) appelle `AdaptiveRate(g, generations)` à chaque génération, puis comparez la trajectoire de convergence (meilleur fitness par génération) à celle du taux fixe $0{,}01$. Hypothèse à vérifier : le taux adaptatif doit atteindre un meilleur optimum final sur Rastrigin, en explorant largement au début puis en stabilisant la population.

In [13]:
// Exercice 3 — Mutation adaptative (étudiant à compléter)
// Surchagez Run pour appeler MutateGauss avec AdaptiveRate(g, generations), comparez la trajectoire.
"Exercice 3 à compléter : brancher AdaptiveRate dans Run, comparer la convergence au taux fixe.".Display();


Exercice 3 à compléter : brancher AdaptiveRate dans Run, comparer la convergence au taux fixe.

***

## Conclusion

Vous avez implémenté **from-scratch** un algorithme génétique générique en C#, validé sur deux problèmes-test opposés (OneMax binaire, Rastrigin continu), et étudié deux leviers-clés (élitisme, mutation adaptatif). Leçon centrale : un GA n'est pas une boîte noire — c'est **sélection + crossover + mutation** assemblés sur une population, et .NET n'a pas besoin de framework GA pour l'enseigner.

**Navigation** : [<< Search-4 Local Search](Search-04-LocalSearch.ipynb) | [↑ Série Search (C#)](../README.md) | [Search-6 Adversarial Search >>](Search-06-AdversarialSearch.ipynb)

### Références

- Holland, J.H. (1975). *Adaptation in Natural and Artificial Systems*.
- Goldberg, D.E. (1989). *Genetic Algorithms in Search, Optimization and Machine Learning*.
- Jumeau Python : [Search-05-GeneticAlgorithms](Search-05-GeneticAlgorithms.ipynb) (DEAP + PyGAD).

***

## 13. Pont finance : évolution d'expressions alpha (issue #24004)

Le jumeau Python de ce carnet (tranche 1, issue #23998) a transformé sa grammaire GP en moteur d'alpha mining sur un panier d'actions réel. Cette section réalise la traversée côté C# : les expressions sont des **arbres immuables** (hierarchie de `record` C#) évolutés par l'organe `GeneticAlgorithm<T>` de la section 2 — pas par une bibliothèque GP externe — et le verdict est lu sur les **propres runs C#**, jamais recopié du jumeau.

Le protocole disciplinaire est identique (doctrine §C du dépôt) : panier de données gelé, features laguées d'un jour, moteur de portefeuille dollar-neutre net de frais, validation walk-forward 5 plis × 5 graines, conjonction edge ≥ 2σ **et** Diebold-Mariano p < 0,05, verdict honnête — jamais « promising ».

In [14]:
// --- Section 13 : panier gele, features et moteur de portefeuille (issue #24004) ---
using System.Globalization;

// Porteur statique cross-cellules : les membres publics d'un type sont visibles
// des submissions suivantes, y compris depuis les methodes des classes des cellules ulterieures.
public static class Alpha13
{
    public static string[] Tickers = Array.Empty<string>();
    public static DateTime[] Dates = Array.Empty<DateTime>();
    public static int NDays = 0, NTicks = 0, N = 0, FirstD = 0, LastD = 0;
    public static double[][] Feats = Array.Empty<double[]>();   // 6 features par obs
    public static double[] RetNext = Array.Empty<double>();     // rendement reponde t+1
    public static string[] FeatNames = { "r1", "r5", "r21", "mom", "vol", "volr" };
}

var path13 = "data/equity_panel_daily.csv";
var lines = System.IO.File.ReadAllLines(path13);
var heads = lines[0].Split(',');
Alpha13.Tickers = heads.Skip(1).ToArray();
int nD = lines.Length - 1, nT = Alpha13.Tickers.Length;
var dates = new DateTime[nD];
var closes = new double[nT][];
for (int t = 0; t < nT; t++) closes[t] = new double[nD];
for (int d = 0; d < nD; d++)
{
    var p = lines[d + 1].Split(',');
    dates[d] = DateTime.Parse(p[0], CultureInfo.InvariantCulture);
    for (int t = 0; t < nT; t++) closes[t][d] = double.Parse(p[t + 1], CultureInfo.InvariantCulture);
}
Alpha13.Dates = dates;

// Rendements quotidiens par titre
var rets = new double[nT][];
for (int t = 0; t < nT; t++)
{
    rets[t] = new double[nD];
    for (int d = 1; d < nD; d++) rets[t][d] = closes[t][d] / closes[t][d - 1] - 1.0;
}

// std d'echantillon (ddof=1, comme pandas rolling.std) des 63 rendements finissant en j
double Std63(int t, int j)
{
    double m = 0; for (int k = j - 62; k <= j; k++) m += rets[t][k];
    m /= 63.0;
    double s = 0; for (int k = j - 62; k <= j; k++) { double e = rets[t][k] - m; s += e * e; }
    return Math.Sqrt(s / 62.0);
}

// Fenetre valide : mom (c[d-1]/c[d-253]) exige d >= 253 ; ret_next exige d <= nD-2.
int firstD = 253, lastD = nD - 2;
int nDays = lastD - firstD + 1, N = nDays * nT;
var feats = new double[6][];
for (int f = 0; f < 6; f++) feats[f] = new double[N];
var retNext = new double[N];
for (int d = firstD; d <= lastD; d++)
{
    int day = d - firstD;
    for (int t = 0; t < nT; t++)
    {
        int o = day * nT + t;
        feats[0][o] = rets[t][d - 1];                                   // r1 : hier
        feats[1][o] = closes[t][d - 1] / closes[t][d - 6] - 1.0;        // r5 lagge
        feats[2][o] = closes[t][d - 1] / closes[t][d - 22] - 1.0;       // r21 lagge
        feats[3][o] = closes[t][d - 1] / closes[t][d - 253] - 1.0;      // mom 12-1 lagge
        feats[4][o] = Std63(t, d - 1) * Math.Sqrt(252.0);               // vol annualisee laggee
        feats[5][o] = Std63(t, d - 1) / Std63(t, d - 64) - 1.0;         // ratio de vol lagge
        retNext[o] = closes[t][d + 1] / closes[t][d] - 1.0;             // reponde t -> t+1
    }
}
Alpha13.Feats = feats; Alpha13.RetNext = retNext;
Alpha13.NDays = nDays; Alpha13.NTicks = nT; Alpha13.N = N; Alpha13.FirstD = firstD; Alpha13.LastD = lastD;

var sb = new System.Text.StringBuilder();
sb.AppendLine($"Panier ({nT} titres, hors FAANG/Mag7) : {string.Join(", ", Alpha13.Tickers)}");
sb.AppendLine($"Periode : {dates[0]:yyyy-MM-dd} -> {dates[nD - 1]:yyyy-MM-dd}  ({nD} jours de bourse)");
sb.AppendLine($"Panel : {N} observations (Date x Ticker), 6 features");
sb.AppendLine();
sb.AppendLine($"{"Date",12} {"Ticker",6} {"r1",8} {"r5",8} {"r21",8} {"mom",8} {"vol",8} {"volr",8} {"ret_next",9}");
int lastDay = nDays - 1;
for (int t = 0; t < nT; t++)
{
    int o = lastDay * nT + t;
    sb.AppendLine($"{dates[lastD]:yyyy-MM-dd} {Alpha13.Tickers[t],6} {feats[0][o],8:F4} {feats[1][o],8:F4} {feats[2][o],8:F4} {feats[3][o],8:F4} {feats[4][o],8:F4} {feats[5][o],8:F4} {retNext[o],9:F4}");
}
sb.ToString().Display();

Panier (8 titres, hors FAANG/Mag7) : KO, XOM, JPM, PG, CAT, BA, CVX, WMT
Periode : 2016-01-04 -> 2026-10-30  (2723 jours de bourse)
Panel : 19752 observations (Date x Ticker), 6 features

        Date Ticker       r1       r5      r21      mom      vol     volr  ret_next
2026-10-29     KO   0,0060  -0,0103   0,0563   0,3441   0,1673  -0,3060    0,0136
2026-10-29    XOM   0,0117   0,0221   0,1281   0,6155   0,2414  -0,1637    0,0216
2026-10-29    JPM   0,0105   0,0123   0,0179   0,1371   0,1538  -0,3456   -0,0190
2026-10-29     PG  -0,0061   0,0157   0,0741   0,0732   0,1975  -0,1545    0,0061
2026-10-29    CAT   0,0001  -0,0340  -0,1350   0,3650   0,3292  -0,3504   -0,0381
2026-10-29     BA  -0,0062  -0,1307  -0,1094  -0,2505   0,3755   0,0330   -0,0064
2026-10-29    CVX   0,0043   0,0107   0,1214   0,5342   0,2330  -0,1039    0,0175
2026-10-29    WMT  -0,0046   0,0082   0,1121   0,1465   0,2758  -0,0074    0,0186


**Lecture du panel.** Le panier gelé `data/equity_panel_daily.csv` couvre **8 titres hors FAANG/Mag7** (KO, XOM, JPM, PG, CAT, BA, CVX, WMT) du 2016-01-04 au 2026-10-30, soit **2 723 jours de bourse**, reconstruits en **19 752 observations** (jour × titre) porteuses de 6 features et du rendement réponse. Toutes les features sont **laguées d'un jour** : `r1` lit hier, `r5`/`r21`/`mom` ferment sur hier (`c[d-1]`), la volatilité annualisée et son ratio ferment sur hier également — le poids du jour t ne voit que le passé, et s'applique au rendement t → t+1. **Aucune fuite d'information par construction**, à l'identique du jumeau Python : le carnet relit le CSV gelé, jamais le réseau.

La dernière ligne du tableau le montre sur un cas concret : le 2026-10-29, CAT affiche un momentum 12-1 de +0,3650 mais un `r5` de −0,1350 — un titre qui a fortement monté sur l'année et qui corrigeait sur la semaine. C'est exactement le genre d'hétérogénéité en coupe qu'un facteur fixe ne peut pas combiner, et qu'une expression apprise peut tenter de combiner.

In [15]:
// --- Moteur de portefeuille : scores -> poids dollar-neutres -> PnL net de frais ---
public static class Engine13
{
    public const double COST_BPS = 5.0;      // frais 5 bps par cote (doctrine section C)
    public const double GROSS = 1.0;         // exposition brute |somme des poids| = 1
    public const int REBAL = 5;              // jours de bourse entre deux rotations
    public const int N_FEATS = 6;

    // Poids dollar-neutres depuis des scores : z-score en coupe date par date,
    // normalises a l'exposition brute GROSS, re-equilibres tous les REBAL jours
    // (portes entre deux rotations). Dispersion quasi nulle (expression constante) :
    // aucun signal en coupe -> position plate, jamais un panier equipondere de bruit.
    public static double[] Weights(double[] scores)
    {
        int nT = Alpha13.NTicks, nDays = Alpha13.NDays;
        var w = new double[Alpha13.N];
        var carried = new double[nT];
        var z = new double[nT];
        for (int day = 0; day < nDays; day++)
        {
            if (day % REBAL == 0)
            {
                int o0 = day * nT;
                double mean = 0; for (int t = 0; t < nT; t++) mean += scores[o0 + t];
                mean /= nT;
                double ss = 0; for (int t = 0; t < nT; t++) { double e = scores[o0 + t] - mean; ss += e * e; }
                double std = Math.Sqrt(ss / (nT - 1));
                if (std < 1e-9)
                {
                    for (int t = 0; t < nT; t++) carried[t] = 0.0;
                }
                else
                {
                    double absZ = 0;
                    for (int t = 0; t < nT; t++) { z[t] = (scores[o0 + t] - mean) / std; absZ += Math.Abs(z[t]); }
                    double denom = absZ / GROSS + 1e-12;
                    for (int t = 0; t < nT; t++) carried[t] = z[t] / denom;
                }
            }
            for (int t = 0; t < nT; t++) w[day * nT + t] = carried[t];
        }
        return w;
    }

    // PnL quotidien net de frais : somme poids * rendement t+1 - cout de rotation.
    public static double[] DailyPnl(double[] w)
    {
        int nT = Alpha13.NTicks, nDays = Alpha13.NDays;
        var pnl = new double[nDays];
        var lastW = new double[nT];
        for (int day = 0; day < nDays; day++)
        {
            double gross = 0, turn = 0;
            for (int t = 0; t < nT; t++)
            {
                int o = day * nT + t;
                gross += w[o] * Alpha13.RetNext[o];
                turn += Math.Abs(w[o] - lastW[t]);
                lastW[t] = w[o];
            }
            pnl[day] = gross - turn * (COST_BPS / 10000.0);
        }
        return pnl;
    }

    // Sharpe annualise d'une serie de PnL quotidiens (ecart-type d'echantillon).
    public static double Sharpe(double[] pnl, int fromDay = 0, int toDayExcl = -1)
    {
        if (toDayExcl < 0) toDayExcl = pnl.Length;
        int n = toDayExcl - fromDay;
        if (n <= 1) return 0.0;
        double m = 0; for (int i = fromDay; i < toDayExcl; i++) m += pnl[i];
        m /= n;
        double s = 0; for (int i = fromDay; i < toDayExcl; i++) { double e = pnl[i] - m; s += e * e; }
        double std = Math.Sqrt(s / (n - 1));
        return std == 0 ? 0.0 : m / std * Math.Sqrt(252.0);
    }
}

// --- Facteurs fixes : memes features, aucun parametre appris ---
double[] PnlFromFeat(int featIdx, bool negate = false)
{
    var scores = new double[Alpha13.N];
    for (int o = 0; o < Alpha13.N; o++) scores[o] = negate ? -Alpha13.Feats[featIdx][o] : Alpha13.Feats[featIdx][o];
    return Engine13.DailyPnl(Engine13.Weights(scores));
}
var pnlMom = PnlFromFeat(3);                                  // momentum 12-1
var pnlVolInv = PnlFromFeat(4, negate: true);                 // vol-inverse
double[] pnlEq;
{
    var wEq = new double[Alpha13.N];                          // exposition brute 1 : 1/8 par titre
    for (int o = 0; o < Alpha13.N; o++) wEq[o] = 1.0 / Alpha13.NTicks;
    pnlEq = Engine13.DailyPnl(wEq);
}
double Cum(double[] pnl) { double c = 1.0; foreach (var r in pnl) c *= 1.0 + r; return c - 1.0; }

var sb2 = new System.Text.StringBuilder();
sb2.AppendLine($"Facteurs fixes sur la periode complete (meme moteur, meme frais, re-equilibre {Engine13.REBAL} j) :");
sb2.AppendLine($"{"strategie",-22} {"Sharpe",10} {"PnL cumule",12}");
sb2.AppendLine($"{"momentum 12-1",-22} {Engine13.Sharpe(pnlMom),10:F3} {Cum(pnlMom),12:F4}");
sb2.AppendLine($"{"vol-inverse",-22} {Engine13.Sharpe(pnlVolInv),10:F3} {Cum(pnlVolInv),12:F4}");
sb2.AppendLine($"{"panier equipondere",-22} {Engine13.Sharpe(pnlEq),10:F3} {Cum(pnlEq),12:F4}");
sb2.ToString().Display();

Facteurs fixes sur la periode complete (meme moteur, meme frais, re-equilibre 5 j) :
strategie                  Sharpe   PnL cumule
momentum 12-1               0,256       0,2879
vol-inverse                -0,192      -0,3271
panier equipondere          0,897       3,3696


**Lecture des baselines.** Les trois facteurs fixes passent par le même moteur — mêmes frais de 5 bps par unité de rotation, même rééquilibrage tous les 5 jours — que les expressions évolutées de la suite : la comparaison est donc honnête. Sur la période complète, le momentum 12-1 fait **Sharpe 0,256** (+28,8 % cumulé), la vol-inverse **−0,192** (−32,7 %), et le panier équipondéré **0,897** (+337,0 %) : la décennie a porté le beta bien plus que la rotation sectorielle, ce qui fixe une barre sévère — battre le momentum ne suffit pas, il faudrait encore rattraper un panier long qui a presque quadruplé. Ces trois chiffres sont **bit-identiques à ceux du jumeau Python** (mêmes données, même moteur réimplémenté) : la parité de l'infrastructure est vérifiée avant même de faire évoluer la moindre expression.

Un détail d'implémentation mérite d'être noté : la garde `std < 1e-9`. Avec des scores constants (une expression réduite à une constante), le z-score en coupe devrait être nul partout — position plate. Mais l'accumulation séquentielle C# du moyenneur laisse une poussière flottante d'ordre 1e-16, que la normalisation transformerait en poids ±1/8 uniformes : **un panier équipondéré déguisé**. La garde court-circuite vers la position plate. Le jumeau Python n'en a pas besoin — pandas tombe exactement à zéro en vectorisé — c'est un idiome propre à la boucle C#, documenté dans le `known_differences` de la paire.

In [16]:
// --- Expression alpha : grammaire a arbres, branchee sur l'organe GeneticAlgorithm<T> ---
public abstract record GpNode;
public record GpAdd(GpNode L, GpNode R) : GpNode;
public record GpSub(GpNode L, GpNode R) : GpNode;
public record GpMul(GpNode L, GpNode R) : GpNode;
public record GpDiv(GpNode L, GpNode R) : GpNode;   // division protegee
public record GpNeg(GpNode A) : GpNode;
public record GpAbs(GpNode A) : GpNode;
public record GpFeat(int I) : GpNode;               // terminal feature
public record GpConst(double V) : GpNode;           // constante ephemere ~ U(0.25, 2.0)

public static class Gp13
{
    public const int MAX_DEPTH = 4;
    public const double COMPLEXITY_PENALTY = 0.02;  // par noeud : favorise les expressions courtes

    public static GpNode RandomTree(Random r, int maxDepth)
    {
        if (maxDepth <= 0 || r.NextDouble() < 0.25)
            return r.NextDouble() < 0.8
                ? new GpFeat(r.Next(Engine13.N_FEATS))
                : new GpConst(Math.Round(0.25 + r.NextDouble() * 1.75, 2));
        int op = r.Next(6);
        var l = RandomTree(r, maxDepth - 1);
        return op switch
        {
            0 => new GpAdd(l, RandomTree(r, maxDepth - 1)),
            1 => new GpSub(l, RandomTree(r, maxDepth - 1)),
            2 => new GpMul(l, RandomTree(r, maxDepth - 1)),
            3 => new GpDiv(l, RandomTree(r, maxDepth - 1)),
            4 => new GpNeg(l),
            _ => new GpAbs(l),
        };
    }

    public static int Count(GpNode n) => n switch
    {
        GpAdd a => 1 + Count(a.L) + Count(a.R),
        GpSub a => 1 + Count(a.L) + Count(a.R),
        GpMul a => 1 + Count(a.L) + Count(a.R),
        GpDiv a => 1 + Count(a.L) + Count(a.R),
        GpNeg a => 1 + Count(a.A),
        GpAbs a => 1 + Count(a.A),
        _ => 1,
    };

    public static double Eval(GpNode n, int o)
    {
        switch (n)
        {
            case GpAdd a: return Eval(a.L, o) + Eval(a.R, o);
            case GpSub a: return Eval(a.L, o) - Eval(a.R, o);
            case GpMul a: return Eval(a.L, o) * Eval(a.R, o);
            case GpDiv a:
                double b = Eval(a.R, o);
                return Math.Abs(b) > 1e-9 ? Eval(a.L, o) / b : 0.0;
            case GpNeg a: return -Eval(a.A, o);
            case GpAbs a: return Math.Abs(Eval(a.A, o));
            case GpFeat f: return Alpha13.Feats[f.I][o];
            case GpConst c: return c.V;
            default: return 0.0;
        }
    }

    public static double[] EvalAll(GpNode n)
    {
        var s = new double[Alpha13.N];
        for (int o = 0; o < Alpha13.N; o++) s[o] = Eval(n, o);
        return s;
    }

    // Fitness : Sharpe net de frais sur la fenetre [0, maxDayExcl) - penalite de complexite.
    public static double Fitness(GpNode n, int maxDayExcl)
    {
        var scores = EvalAll(n);
        for (int o = 0; o < Alpha13.N; o++)
            if (double.IsNaN(scores[o]) || double.IsInfinity(scores[o])) return -10.0;
        var pnl = Engine13.DailyPnl(Engine13.Weights(scores));
        return Engine13.Sharpe(pnl, 0, maxDayExcl) - COMPLEXITY_PENALTY * Count(n);
    }

    public static string Pretty(GpNode n) => n switch
    {
        GpAdd a => $"add({Pretty(a.L)}, {Pretty(a.R)})",
        GpSub a => $"sub({Pretty(a.L)}, {Pretty(a.R)})",
        GpMul a => $"mul({Pretty(a.L)}, {Pretty(a.R)})",
        GpDiv a => $"div({Pretty(a.L)}, {Pretty(a.R)})",
        GpNeg a => $"neg({Pretty(a.A)})",
        GpAbs a => $"abs({Pretty(a.A)})",
        GpFeat f => Alpha13.FeatNames[f.I],
        GpConst c => c.V.ToString("0.##", System.Globalization.CultureInfo.InvariantCulture),
        _ => "?",
    };

    // Sous-arbre au rang target dans un parcours pre-ordre (null si introuvable).
    static GpNode GetSubtree(GpNode n, int target, ref int i)
    {
        if (i++ == target) return n;
        return n switch
        {
            GpAdd a => GetSubtree(a.L, target, ref i) ?? GetSubtree(a.R, target, ref i),
            GpSub a => GetSubtree(a.L, target, ref i) ?? GetSubtree(a.R, target, ref i),
            GpMul a => GetSubtree(a.L, target, ref i) ?? GetSubtree(a.R, target, ref i),
            GpDiv a => GetSubtree(a.L, target, ref i) ?? GetSubtree(a.R, target, ref i),
            GpNeg a => GetSubtree(a.A, target, ref i),
            GpAbs a => GetSubtree(a.A, target, ref i),
            _ => null,
        };
    }

    // Reconstruit l'arbre avec repl en position target (pre-ordre).
    static GpNode ReplaceAt(GpNode n, int target, GpNode repl, int i = 0)
    {
        if (i == target) return repl;
        switch (n)
        {
            case GpAdd a: return new GpAdd(ReplaceAt(a.L, target, repl, i + 1), ReplaceAt(a.R, target, repl, i + 1 + Count(a.L)));
            case GpSub a: return new GpSub(ReplaceAt(a.L, target, repl, i + 1), ReplaceAt(a.R, target, repl, i + 1 + Count(a.L)));
            case GpMul a: return new GpMul(ReplaceAt(a.L, target, repl, i + 1), ReplaceAt(a.R, target, repl, i + 1 + Count(a.L)));
            case GpDiv a: return new GpDiv(ReplaceAt(a.L, target, repl, i + 1), ReplaceAt(a.R, target, repl, i + 1 + Count(a.L)));
            case GpNeg a: return new GpNeg(ReplaceAt(a.A, target, repl, i + 1));
            case GpAbs a: return new GpAbs(ReplaceAt(a.A, target, repl, i + 1));
            default: return n;
        }
    }

    public static (GpNode, GpNode) Crossover(Random r, GpNode a, GpNode b)
    {
        int ia = r.Next(Count(a)), ib = r.Next(Count(b));
        int i1 = 0, i2 = 0;
        var subB = GetSubtree(b, ib, ref i1);
        var subA = GetSubtree(a, ia, ref i2);
        return (ReplaceAt(a, ia, subB), ReplaceAt(b, ib, subA));
    }

    public static GpNode Mutate(Random r, GpNode t)
    {
        int i = r.Next(Count(t));
        return ReplaceAt(t, i, RandomTree(r, 2));
    }
}

var sample13 = Gp13.RandomTree(new Random(7), Gp13.MAX_DEPTH);
var sb3 = new System.Text.StringBuilder();
sb3.AppendLine($"Grammaire : 6 primitives (add/sub/mul/div/neg/abs), {Engine13.N_FEATS} terminals (features), constantes ~ U(0.25, 2.0), profondeur max {Gp13.MAX_DEPTH}");
sb3.AppendLine($"Fitness = Sharpe net de frais (frais {Engine13.COST_BPS} bps, exposition brute {Engine13.GROSS}) - {Gp13.COMPLEXITY_PENALTY} par noeud");
sb3.AppendLine($"Exemple d'expression valide : {Gp13.Pretty(sample13)}  ({Gp13.Count(sample13)} noeuds)");
sb3.ToString().Display();

Grammaire : 6 primitives (add/sub/mul/div/neg/abs), 6 terminals (features), constantes ~ U(0.25, 2.0), profondeur max 4
Fitness = Sharpe net de frais (frais 5 bps, exposition brute 1) - 0,02 par noeud
Exemple d'expression valide : abs(add(neg(1.73), mul(mul(volr, vol), abs(vol))))  (10 noeuds)


**Lecture de la grammaire.** L'expression alpha est une hiérarchie de `record` immuables : `GpAdd`, `GpSub`, `GpMul`, `GpDiv` (division protégée : dénominateur de module ≤ 1e-9 renvoie 0), `GpNeg`, `GpAbs`, plus deux terminaux — une feature (`GpFeat`) ou une constante éphémère tirée à U(0.25, 2.0). L'immuabilité change l'écriture des opérateurs génétiques : le **crossover** échange des sous-arbres par parcours pré-ordre (`GetSubtree` au rang tiré, `ReplaceAt` reconstruit l'arbre autour) et la **mutation** remplace un sous-arbre par un arbre frais de profondeur 2 — jamais de modification en place, chaque opérateur retourne un nouvel arbre. C'est l'idiome C# natif là où le jumeau Python délègue à `deap.PrimitiveSet` l'énumération et au clonage la sécurité des croisements.

La fitness est le **Sharpe net de frais sur la fenêtre d'entraînement**, moins une pénalité de 0,02 par nœud : à Sharpe égal, l'expression la plus courte gagne — un garde-fou doux contre la mémorisation. La profondeur est plafonnée à 4, ce qui borne le nombre de nœuds (et donc l'espace de recherche) à quelques dizaines : suffisant pour combiner deux ou trois features, insuffisant pour encoder du bruit idiosyncratique. Toute expression qui produirait un score non fini sur une seule observation est rejetée à −10 : le moteur ne voit jamais de NaN.

In [17]:
// --- Evolution d'expressions alpha avec l'organe GeneticAlgorithm<T> (demo, graine 42) ---
const int SEED_DEMO = 42;
int TRAIN_MAX_DAY = Alpha13.NDays - 504;   // ~2 ans de bourse reserves au test

int Tournament3(Random r, Individu<GpNode>[] pop)
{
    int best = r.Next(pop.Length);
    for (int k = 0; k < 2; k++)
    {
        int c = r.Next(pop.Length);
        if (pop[c].Fitness > pop[best].Fitness) best = c;
    }
    return best;
}
(GpNode, GpNode) CxAlpha(Random r, GpNode a, GpNode b)
    => r.NextDouble() < 0.5 ? Gp13.Crossover(r, a, b) : (a, b);
GpNode MutAlpha(Random r, GpNode c, double rate)
    => r.NextDouble() < rate ? Gp13.Mutate(r, c) : c;

var swDemo = System.Diagnostics.Stopwatch.StartNew();
var gaDemo = new GeneticAlgorithm<GpNode>(
    n => Gp13.Fitness(n, TRAIN_MAX_DAY),
    r => Gp13.RandomTree(r, Gp13.MAX_DEPTH),
    CxAlpha, MutAlpha, Tournament3,
    populationSize: 60, mutationRate: 0.2, seed: SEED_DEMO);
var finalDemo = gaDemo.Run(generations: 25);
swDemo.Stop();
var champion = finalDemo.OrderByDescending(p => p.Fitness).First();

var sb4 = new System.Text.StringBuilder();
sb4.AppendLine($"=== Evolution d'expressions alpha (pop=60, 25 generations, graine {SEED_DEMO}) ===");
sb4.AppendLine($"Champion (fitness = {champion.Fitness:F3}) : {Gp13.Pretty(champion.Chromosome)}  ({Gp13.Count(champion.Chromosome)} noeuds)");
sb4.AppendLine($"Temps : {swDemo.Elapsed.TotalSeconds:F1} s");
sb4.AppendLine();
sb4.AppendLine("Generation | Fitness max");
sb4.AppendLine("-----------|------------");
foreach (var g in new[] { 0, 5, 10, 15, 20, 25 })
    sb4.AppendLine($"{g,10} | {gaDemo.HistoryBest[g],11:F3}");
// Sharpe OOS du champion sur les 2 ans reserves
var pnlChampDemo = Engine13.DailyPnl(Engine13.Weights(Gp13.EvalAll(champion.Chromosome)));
sb4.AppendLine();
sb4.AppendLine($"Sharpe OOS du champion (2 ans reserves) : {Engine13.Sharpe(pnlChampDemo, TRAIN_MAX_DAY):F3}  -- a comparer au fitness train {champion.Fitness:F3}");
sb4.ToString().Display();

=== Evolution d'expressions alpha (pop=60, 25 generations, graine 42) ===
Champion (fitness = 0,657) : div(r21, add(0.56, r21))  (5 noeuds)
Temps : 7,5 s

Generation | Fitness max
-----------|------------
         0 |       0,477
         5 |       0,657
        10 |       0,657
        15 |       0,657
        20 |       0,657
        25 |       0,657

Sharpe OOS du champion (2 ans reserves) : -0,501  -- a comparer au fitness train 0,657


**Lecture de la démo.** L'organe `GeneticAlgorithm<T>` de la section 2 — inchangé, celui-là même qui a résolu OneMax et Rastrigin — fait évoluer des arbres d'expression : la sélection est un tournoi à 3 (l'équivalent du `selTournament(k=3)` du jumeau Python, ici en six lignes), le crossover échange des sous-arbres avec probabilité 0,5, la mutation frappe à taux 0,2. En 25 générations (6 à 7 s sur ce panier), la population converge de 0,477 à **0,657** et le champion est `div(r21, add(0.56, r21))` — cinq nœuds : une mise en forme non linéaire du momentum d'un mois, bornée entre −0,56/1,56 environ, qui aplatit les grands `r21` et amplifie les petits. La pénalité de complexité a fait son travail : rien de profond n'a survécu.

La dernière ligne est la leçon de la section : ce champion, **fitness 0,657 en entraînement, signe un Sharpe de −0,501 sur les deux ans réservés**. La formule n'était pas « vraie » — elle était la mieux ajustée à ce tronçon particulier du passé. C'est le surapprentissage de sélection : avec des centaines d'expressions évaluées, la meilleure en échantillon est presque mécaniquement celle qui a capturé du bruit. Une seule démo ne prouve rien ; la cellule suivante applique le protocole complet qui décide.

In [18]:
// --- Verdict doctrine section C : walk-forward 5 plis x 5 graines, edge 2-sigma + Diebold-Mariano ---
int[] SEEDS_C = { 0, 1, 7, 42, 99 };
const int N_FOLDS = 5;
int oosStart = Alpha13.NDays - 504, oosEnd = Alpha13.NDays;   // jours de panel hors echantillon

// erf (Abramowitz-Stegun 7.1.26, precision ~1.5e-7) : la BCL n'expose pas Erf.
double Erf13(double x)
{
    double t = 1.0 / (1.0 + 0.3275911 * Math.Abs(x));
    double y = 1.0 - (((((1.061405429 * t - 1.453152027) * t) + 1.421413741) * t - 0.284496736) * t + 0.254829592) * t * Math.Exp(-x * x);
    return x >= 0 ? y : -y;
}

// Diebold-Mariano bilateral, variance Newey-West (retard 1).
// lossA / lossB : series de pertes quotidiennes (perte = -PnL net).
double DmPvalue(double[] lossA, double[] lossB)
{
    int n = Math.Min(lossA.Length, lossB.Length);
    var d = new double[n];
    double dmean = 0;
    for (int i = 0; i < n; i++) { d[i] = lossA[i] - lossB[i]; dmean += d[i]; }
    dmean /= n;
    double g0 = 0; for (int i = 0; i < n; i++) { double e = d[i] - dmean; g0 += e * e; }
    g0 /= n;
    double g1 = 0; for (int i = 1; i < n; i++) g1 += (d[i] - dmean) * (d[i - 1] - dmean);
    g1 /= n;
    double varD = (g0 + 2.0 * g1) / n;
    if (varD <= 0) return 1.0;
    double stat = dmean / Math.Sqrt(varD);
    return 2.0 * (1.0 - 0.5 * (1.0 + Erf13(Math.Abs(stat) / Math.Sqrt(2.0))));
}

// perte quotidienne = -PnL (le DM compare des series de pertes).
double[] Neg(List<double> x)
{
    var a = new double[x.Count];
    for (int i = 0; i < a.Length; i++) a[i] = -x[i];
    return a;
}

GpNode EvolveChampion(int maxDayExcl, int seed, int ngen = 20, int popSize = 50)
{
    int T3(Random r, Individu<GpNode>[] pop)
    {
        int best = r.Next(pop.Length);
        for (int k = 0; k < 2; k++) { int c = r.Next(pop.Length); if (pop[c].Fitness > pop[best].Fitness) best = c; }
        return best;
    }
    var ga = new GeneticAlgorithm<GpNode>(
        n => Gp13.Fitness(n, maxDayExcl),
        r => Gp13.RandomTree(r, Gp13.MAX_DEPTH),
        (r, a, b) => r.NextDouble() < 0.5 ? Gp13.Crossover(r, a, b) : (a, b),
        (r, c, rate) => r.NextDouble() < rate ? Gp13.Mutate(r, c) : c,
        T3, populationSize: popSize, mutationRate: 0.2, seed: seed);
    var finalPop = ga.Run(ngen);
    return finalPop.OrderByDescending(p => p.Fitness).First().Chromosome;
}

var swC = System.Diagnostics.Stopwatch.StartNew();
var rowsC = new List<(int seed, int foldStart, double sharpeOos, string expr)>();
var edgesC = new Dictionary<int, double>();
var dmPsC = new Dictionary<int, double>();
foreach (int seed in SEEDS_C)
{
    var evoOos = new List<double>(); var momOos = new List<double>();
    int foldLen = (oosEnd - oosStart + N_FOLDS - 1) / N_FOLDS;
    for (int k = 0; k < N_FOLDS; k++)
    {
        int fs = oosStart + k * foldLen;
        int fe = Math.Min(fs + foldLen, oosEnd);
        var champ = EvolveChampion(fs, seed);                 // entraine uniquement sur le passe du pli
        var pnlChamp = Engine13.DailyPnl(Engine13.Weights(Gp13.EvalAll(champ)));
        for (int day = fs; day < fe; day++) { evoOos.Add(pnlChamp[day]); momOos.Add(pnlMom[day]); }
        rowsC.Add((seed, fs, Engine13.Sharpe(pnlChamp, fs, fe), Gp13.Pretty(champ)));
    }
    double edge = 0; for (int i = 0; i < evoOos.Count; i++) edge += evoOos[i] - momOos[i];
    edgesC[seed] = edge / evoOos.Count * 10000.0;
    dmPsC[seed] = DmPvalue(Neg(evoOos), Neg(momOos));
}
swC.Stop();

double edgeMean = edgesC.Values.Average();
double edgeSd = 0;
foreach (var v in edgesC.Values) edgeSd += (v - edgeMean) * (v - edgeMean);
edgeSd = Math.Sqrt(edgeSd / (edgesC.Count - 1));
double edgeSe = edgeSd / Math.Sqrt(edgesC.Count);
double dmMedian = dmPsC.Values.OrderBy(v => v).ElementAt(dmPsC.Count / 2);

var sb5 = new System.Text.StringBuilder();
sb5.AppendLine($"Champions par pli (5 graines x 5 plis, {swC.Elapsed.TotalSeconds:F0} s au total) -- extraits :");
foreach (var r in rowsC.Take(3).Concat(rowsC.Skip(rowsC.Count - 2)))
    sb5.AppendLine($"  graine {r.seed,2}  pli jour {r.foldStart,4}  Sharpe OOS {r.sharpeOos,7:F3}  {r.expr.Substring(0, Math.Min(58, r.expr.Length))}");
sb5.AppendLine();
sb5.AppendLine("Edge quotidien vs momentum 12-1 (bps), par graine : {" +
    string.Join(", ", edgesC.OrderBy(kv => kv.Key).Select(kv => $"{kv.Key}: {kv.Value:F2}")) + "}");
sb5.AppendLine($"Edge moyen : {edgeMean:+0.00;-0.00} bps/jour (SE inter-graines {edgeSe:F2}, seuil 2-sigma = +/-{2 * edgeSe:F2})");
sb5.AppendLine($"DM p-valeur mediane (5 graines) : {dmMedian:F3}");
string verdict = "INCONCLUSIVE";
if (edgeMean >= 2 * edgeSe && dmMedian < 0.05) verdict = "BEATS";
else if (edgeMean <= -2 * edgeSe && dmMedian < 0.05) verdict = "NO BEATS (dominee)";
sb5.AppendLine();
sb5.AppendLine($"VERDICT (vs momentum 12-1) : {verdict}");
sb5.ToString().Display();

Champions par pli (5 graines x 5 plis, 145 s au total) -- extraits :
  graine  0  pli jour 1965  Sharpe OOS  -0,440  abs(add(abs(mom), volr))
  graine  0  pli jour 2066  Sharpe OOS   1,557  abs(mom)
  graine  0  pli jour 2167  Sharpe OOS   0,379  abs(add(r21, abs(sub(mom, r5))))
  graine 99  pli jour 2268  Sharpe OOS   1,156  abs(add(r21, add(r21, abs(mom))))
  graine 99  pli jour 2369  Sharpe OOS  -0,311  add(abs(mom), r21)

Edge quotidien vs momentum 12-1 (bps), par graine : {0: 1,87, 1: -1,79, 7: -3,73, 42: -1,55, 99: -0,08}
Edge moyen : -1,06 bps/jour (SE inter-graines 0,93, seuil 2-sigma = +/-1,87)
DM p-valeur mediane (5 graines) : 0,326

VERDICT (vs momentum 12-1) : INCONCLUSIVE


**Lecture du verdict.** Le protocole walk-forward ré-évolue un champion par pli, entraîné **uniquement sur les données antérieures au pli** (jamais de sélection rétrospective), et empile les rendements hors échantillon de 5 plis × 5 graines. Les expressions gagnantes sont courtes et lisibles — `abs(mom)`, `add(abs(mom), r21)`, une mise en forme du momentum d'un mois — mais leur Sharpe hors échantillon va de −0,44 à +1,56 selon le pli : aucune n'est stable. L'edge quotidien agrégé contre le momentum 12-1 vaut **−1,06 bps/jour** en moyenne, avec une erreur-type inter-graines de 0,93 : quatre graines sur cinq sont négatives, mais l'intervalle ±1,87 bps (2σ) contient largement zéro. Le Diebold-Mariano médian (Newey-West retard 1) confirme : **p = 0,326**, loin du seuil 0,05.

**VERDICT : INCONCLUSIVE.** Ni supérieur ni significativement inférieur au momentum : la GP sur ce panier, avec cette grammaire et ce budget de calcul, n'achète pas d'edge mesurable. Le jumeau Python aboutit au même verdict sur ses propres runs (issue #23998) — deux réalisations RNG indépendantes, deux dispositifs disciplinaires identiques, la même conclusion prudente : c'est précisément ce que la doctrine appelle un résultat honnête. La leçon de la section n'est pas « la GP ne marche pas », elle est : **un Sharpe d'entraînement n'est pas une preuve**, et seule la conjonction 2σ + Diebold-Mariano sur données hors échantillon sépare un facteur d'un artefact de sélection.

**Bilan du pont finance.** La section a réutilisé chaque pièce du carnet : l'organe `GeneticAlgorithm<T>` (section 2) pour l'évolution, les idiomes `record` immuables pour la grammaire, la discipline expérimentale de la comparaison chiffrée (sections 5-6) pour le verdict. Deux apports propres au C# par rapport au jumeau : la **garde de dispersion** (une expression constante ne peut pas se déguiser en panier équipondéré) et le crossover par reconstruction pré-ordre sur arbres immuables — deux idiomes que la vectorisation pandas rend inutiles côté Python. L'exercice 4 laisse la question ouverte : une grammaire plus riche (`max(a, b)`) achète-t-elle un edge que six primitives arithmétiques n'atteignent pas — ou seulement davantage de surface de surapprentissage ? Réponse attendue par la même voie que celle de cette section : évolution, fenêtre réservée, verdict de la doctrine — pas d'intuition.

In [19]:
// --- Exercice 4 : etendre la grammaire (issue #24004) ---
// Enonce : la grammaire de la section 13 n'a que 6 primitives arithmetiques.
// 1. Ajoutez une primitive conditionnelle max(a, b) au record set (GpMax).
// 2. Faites evoluer une population avec la grammaire etendue sur TRAIN_MAX_DAY.
// 3. Comparez le champion etendu au champion de la demo sur les memes 2 ans
//    hors echantillon (meme moteur, memes frais) : la richesse de la grammaire
//    achete-t-elle un edge, ou seulement du surapprentissage ?
// Indice : mesurez le nombre de noeuds moyen des champions avant / apres.

// TODO etudiant
Console.WriteLine("Exercice a completer : grammaire etendue, evolution, comparaison OOS au champion de la demo");

Exercice a completer : grammaire etendue, evolution, comparaison OOS au champion de la demo


## 14. Un bras LLM dans la même boucle : proposer au lieu de muter

La section 13 fait évoluer des expressions alpha par **mutation et croisement** d'une population, via l'organe `GeneticAlgorithm<T>` du carnet. Un modèle de langue peut tenir l'autre rôle : **proposer** des expressions nouvelles à chaque tour, que la même fonction `Gp13.Fitness` évalue et classe. Rien d'autre ne change — mêmes features, même grammaire, même moteur de portefeuille, même walk-forward, mêmes frais. Cette section est la tranche C# de l'issue #25696, symétrique de la section 14 du jumeau Python (#25365) : le dispositif est répliqué à l'identique, le verdict est lu sur les runs propres de ce carnet.

Ce que la comparaison exige pour être honnête :

| Exigence | Ce que fait cette section |
|---|---|
| **Même grammaire** | Six primitives (`add`, `sub`, `mul`, `div`, `neg`, `abs`), six features, constantes dans `[0.25, 2.0]`, profondeur ≤ 4. Un parseur strict refuse tout le reste — aucun `eval`, aucune primitive hors liste, conversion directe en `GpNode`. |
| **Même budget** | Le budget n'est pas `population × générations`, c'est le **nombre réel d'appels à la fitness** que l'AG a consommés sur ce pli. Le compteur `FitnessCounter14` est unique et les deux bras passent par lui. |
| **Mêmes données** | Le panier gelé de la section 13, le même découpage 5 plis et les mêmes graines (0, 1, 7, 42, 99). |
| **Même verdict** | Edge inter-graines ≥ 2σ **et** Diebold-Mariano `p < 0,05` sur une **perte de précision** (erreur quadratique entre score standardisé et rendement futur standardisé, en coupe), plus un rapport de biais signé par bras. |

**Service et reproductibilité.** Le bras LLM appelle un endpoint **vLLM** nommé dans la cellule suivante (`http://192.168.0.47:5002/v1`, modèle `qwen3.6-35b-a3b`), avec sortie **JSON Schema strict** pour obtenir quarante propositions par appel. Les réponses sont mises en cache **hors du dépôt** : une ré-exécution relit la réponse enregistrée et **refait toutes les évaluations de fitness**, donc le budget reste identique — le cache supprime l'attente réseau, pas la mesure. La clé du service se lit dans l'environnement (`VLLM_API_KEY_MEDIUM`) et n'apparaît nulle part dans le carnet.

**Ce que cette section ne prétend pas.** Un LLM n'« apprend » pas la finance ici : il propose des formules plausibles dans une grammaire fermée, et c'est la fitness qui tranche. Aucune interprétation économique n'est tirée d'une expression gagnante — seule compte la comparaison à budget égal, sur données hors échantillon.

In [20]:
// --- Section 14 : service LLM, grammaire fermee et parseur strict (issue #25696) ---
using System.Globalization;
using System.Net.Http;
using System.Security.Cryptography;
using System.Text;
using System.Text.Json;
using System.Text.RegularExpressions;
using System.IO;

public static class Llm14
{
    public const string BaseUrl = "http://192.168.0.47:5002/v1";
    public const string Model = "qwen3.6-35b-a3b";
    public const int Batch = 40;
    public const int Workers = 6;
    public const double TimeoutSeconds = 300.0;
    public const double WallClockSeconds = 2400.0;
    public const double Temperature = 0.5;
    public const int MaxTokens = 12000;
    public const int MaxRetries = 2;          // un episode de lenteur du service ne doit pas tuer le run
    public const int ExpectedPerCall = 30;    // evaluations neuves attendues par appel (40 proposees, ~80% acceptees)

    // Cache hors depot, espace de cles propre au jumeau CSharp : la cle est le sha256
    // d'un JSON canonique System.Text.Json, dont l'echappement et l'ordre des membres
    // different de json.dumps Python -- un dossier commun ne partagerait rien.
    public static readonly string CacheDir =
        Environment.GetEnvironmentVariable("COURSIA_ALPHA_LLM_CACHE")
        ?? Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile),
                        ".cache", "coursia", "search05-alpha-llm-v1-csharp");

    public static readonly HttpClient Http = new() { Timeout = TimeSpan.FromSeconds(TimeoutSeconds) };

    // Garde de cle au point d'usage : jamais de cle dans le carnet.
    public static string ApiKey()
    {
        var key = Environment.GetEnvironmentVariable("VLLM_API_KEY_MEDIUM");
        if (string.IsNullOrEmpty(key))
            throw new InvalidOperationException(
                "VLLM_API_KEY_MEDIUM requis : charger la cle du service vLLM dans " +
                "l'environnement (jamais dans le carnet), puis re-executer la section.");
        return key;
    }
}

// Schema JSON strict impose au service : candidates[].prefix + rationale, rien d'autre.
public static class AlphaSchema14
{
    public const string Json =
        "{\"type\":\"object\",\"properties\":{\"candidates\":{\"type\":\"array\",\"items\":" +
        "{\"type\":\"object\",\"properties\":{\"prefix\":{\"type\":\"string\"}," +
        "\"rationale\":{\"type\":\"string\"}},\"required\":[\"prefix\",\"rationale\"]," +
        "\"additionalProperties\":false}}},\"required\":[\"candidates\"]," +
        "\"additionalProperties\":false}";

    public static bool Validate(string content, out List<JsonElement> candidates)
    {
        candidates = null;
        try
        {
            using var doc = JsonDocument.Parse(content);
            var root = doc.RootElement;
            if (root.ValueKind != JsonValueKind.Object
                || !root.TryGetProperty("candidates", out var arr)
                || arr.ValueKind != JsonValueKind.Array)
                return false;
            var list = new List<JsonElement>();
            foreach (var item in arr.EnumerateArray())
            {
                if (item.ValueKind != JsonValueKind.Object) return false;
                list.Add(item.Clone());
            }
            candidates = list;
            return true;
        }
        catch (JsonException) { return false; }
    }
}

public static class AlphaParse14
{
    static readonly Dictionary<string, int> Ops = new()
        { ["add"] = 2, ["sub"] = 2, ["mul"] = 2, ["div"] = 2, ["neg"] = 1, ["abs"] = 1 };
    static readonly HashSet<string> Terminals = new(Alpha13.FeatNames);
    static readonly Regex ConstRe = new(
        @"^(?:0\.(?:2[5-9]|[3-9][0-9])|1(?:\.[0-9]{1,2})?|2(?:\.0{1,2})?)$",
        RegexOptions.Compiled);

    // Grammaire prefixee stricte : six primitives, six features, constantes bornees.
    // Tout refus leve FormatException AVANT toute evaluation -- aucune eval dynamique.
    public static GpNode Parse(string expression)
    {
        var tokens = expression.Replace("(", " ( ").Replace(")", " ) ").Replace(",", " , ")
                               .Split(' ', StringSplitOptions.RemoveEmptyEntries);
        int position = 0;
        GpNode ParseOne(int depth)
        {
            if (depth > Gp13.MAX_DEPTH || position >= tokens.Length)
                throw new FormatException("profondeur ou troncature");
            string token = tokens[position++];
            if (Terminals.Contains(token))
                return new GpFeat(Array.IndexOf(Alpha13.FeatNames, token));
            if (ConstRe.IsMatch(token))
                return new GpConst(double.Parse(token, CultureInfo.InvariantCulture));
            if (!Ops.TryGetValue(token, out int arity))
                throw new FormatException("symbole hors grammaire");
            if (position >= tokens.Length || tokens[position] != "(")
                throw new FormatException("parenthese ouvrante absente");
            position++;
            GpNode first = ParseOne(depth + 1);
            if (arity == 1)
            {
                if (position >= tokens.Length || tokens[position] != ")")
                    throw new FormatException("parenthese fermante absente");
                position++;
                return token == "neg" ? new GpNeg(first) : new GpAbs(first);
            }
            if (position >= tokens.Length || tokens[position] != ",")
                throw new FormatException("virgule absente");
            position++;
            GpNode second = ParseOne(depth + 1);
            if (position >= tokens.Length || tokens[position] != ")")
                throw new FormatException("parenthese fermante absente");
            position++;
            return token switch
            {
                "add" => new GpAdd(first, second),
                "sub" => new GpSub(first, second),
                "mul" => new GpMul(first, second),
                _ => new GpDiv(first, second),
            };
        }
        var parsed = ParseOne(0);
        if (position != tokens.Length) throw new FormatException("queue inattendue");
        return parsed;
    }

    // Hauteur en aretes (feuille = 0), pour le double controle de la boucle LLM.
    public static int Depth(GpNode n) => n switch
    {
        GpAdd a => 1 + Math.Max(Depth(a.L), Depth(a.R)),
        GpSub a => 1 + Math.Max(Depth(a.L), Depth(a.R)),
        GpMul a => 1 + Math.Max(Depth(a.L), Depth(a.R)),
        GpDiv a => 1 + Math.Max(Depth(a.L), Depth(a.R)),
        GpNeg a => 1 + Depth(a.A),
        GpAbs a => 1 + Depth(a.A),
        _ => 0,
    };
}

Directory.CreateDirectory(Llm14.CacheDir);

// Controles positif et negatifs : le parseur accepte la grammaire et rien d'autre.
if (Gp13.Pretty(AlphaParse14.Parse("mul(mom, div(r21, 0.25))")) != "mul(mom, div(r21, 0.25))")
    throw new InvalidOperationException("controle positif du parseur echoue");
foreach (var invalid in new[] { "__import__('os')", "mul(mom, 0.1)", "max(mom, vol)", "mul(mom)" })
{
    try { AlphaParse14.Parse(invalid); }
    catch (FormatException) { continue; }
    throw new InvalidOperationException("expression hors grammaire acceptee : " + invalid);
}

var sb14 = new System.Text.StringBuilder();
sb14.AppendLine($"Service : {Llm14.BaseUrl} ; modele : {Llm14.Model} ; {Llm14.Batch} propositions par appel");
sb14.AppendLine($"Grammaire : 6 primitives, {Alpha13.FeatNames.Length} features, constantes [0.25, 2.0], profondeur <= {Gp13.MAX_DEPTH}");
sb14.AppendLine("Parseur : controle positif et quatre refus PASS");
sb14.AppendLine("Cache de reponses : dossier hors depot, surchargeable par COURSIA_ALPHA_LLM_CACHE");
sb14.ToString().Display();

Service : http://192.168.0.47:5002/v1 ; modele : qwen3.6-35b-a3b ; 40 propositions par appel
Grammaire : 6 primitives, 6 features, constantes [0.25, 2.0], profondeur <= 4
Parseur : controle positif et quatre refus PASS
Cache de reponses : dossier hors depot, surchargeable par COURSIA_ALPHA_LLM_CACHE


### Le budget se mesure, il ne se suppose pas

L'AG ne consomme pas `population × générations` évaluations : `GeneticAlgorithm<T>.Run` n'évalue que les individus effectivement produits, et le compte dépend de la graine. Le compteur ci-dessous enveloppe `Gp13.Fitness` **une fois pour toutes** — l'AG de la section 14 (`EvolveChampionCounted`, copie exacte du champion de la section 13 dont seul le canal fitness change) et le bras LLM passent tous deux par lui, donc le budget du second est, par construction, celui du premier. Aucune constante à recopier, aucun risque de désynchronisation.

Deux propriétés du cache méritent d'être dites avant la grille. **Idempotence** : chaque appel au service est identifié par l'empreinte SHA-256 de la requête canonique (URL, modèle, prompt, schéma, température) ; une reprise de run rejoue depuis le cache les réponses déjà mesurées, sans re-payer le service ni ré-échantillonner — les clés se reproduisent naturellement d'un tour à l'autre, l'évolution des champions étant déterministe pour un tour donné. **Espace de clés propre** : la sérialisation canonique est ici `System.Text.Json`, dont l'échappement et l'ordre des membres diffèrent de `json.dumps` côté Python ; le dossier de cache porte donc un suffixe `-csharp`, deux espaces disjoints valant mieux qu'un faux partage entre jumeaux.

La garde horaire (2 400 s par pli) n'est **pas** le budget : elle protège contre un service lent ou une boucle qui ne progresse plus. Le budget, lui, est sémantique — un nombre d'évaluations.

In [21]:
// --- Compteur unique de fitness : les deux bras passent par la meme porte ---
using System.Globalization;
using System.Threading;
using System.IO;
using System.Net.Http;
using System.Security.Cryptography;
using System.Text;
using System.Text.Json;

public static class FitnessCounter14
{
    public static long N = 0;
    public static double On(GpNode n, int maxDayExcl)
    {
        N++;
        return Gp13.Fitness(n, maxDayExcl);
    }
}

public static class AlphaLlm14
{
    // Copie comptee d'EvolveChampion (section 13) : seul le canal fitness change.
    public static GpNode EvolveChampionCounted(int maxDayExcl, int seed, int ngen = 20, int popSize = 50)
    {
        int T3(Random r, Individu<GpNode>[] pop)
        {
            int best = r.Next(pop.Length);
            for (int k = 0; k < 2; k++) { int c = r.Next(pop.Length); if (pop[c].Fitness > pop[best].Fitness) best = c; }
            return best;
        }
        var ga = new GeneticAlgorithm<GpNode>(
            n => FitnessCounter14.On(n, maxDayExcl),
            r => Gp13.RandomTree(r, Gp13.MAX_DEPTH),
            (r, a, b) => r.NextDouble() < 0.5 ? Gp13.Crossover(r, a, b) : (a, b),
            (r, c, rate) => r.NextDouble() < rate ? Gp13.Mutate(r, c) : c,
            T3, populationSize: popSize, mutationRate: 0.2, seed: seed);
        var finalPop = ga.Run(ngen);
        return finalPop.OrderByDescending(p => p.Fitness).First().Chromosome;
    }

    public static string Prompt(int seed, int callIndex, List<(double Value, string Pretty)> champions)
    {
        var context = champions.Count > 0
            ? string.Join("\n", champions.Select(c =>
                string.Format(CultureInfo.InvariantCulture, "{0:F5} : {1}", c.Value, c.Pretty)))
            : "Aucun candidat evalue.";
        return $@"Tu recherches une expression alpha dans une grammaire de programmation genetique.
Retourne exactement {Llm14.Batch} candidats DISTINCTS sous la cle candidates.
Chaque objet porte: prefix (la chaine prefixee) et rationale (une phrase courte en francais).
Primitives autorisees: add(a,b), sub(a,b), mul(a,b), div(a,b), neg(a), abs(a).
Terminaux autorises: {string.Join(", ", Alpha13.FeatNames.OrderBy(x => x, StringComparer.Ordinal))}. Constantes decimales dans [0.25, 2.0].
Forme strictement prefixee, profondeur <= {Gp13.MAX_DEPTH}, aucune autre fonction, aucun texte hors JSON.
Scores d'entrainement des meilleurs candidats a ce stade:
{context}
Appel {callIndex}, graine {seed}: ameliore les meilleurs ou explore une hypothese differente, sans copier.";
    }

    public sealed record AlphaResponse14(string Content, string FinishReason, int CompletionTokens, bool FromCache);

    static string CacheKey(string prompt)
    {
        // JSON canonique a ordre de membres fixe, echappement System.Text.Json.
        var canon = new StringBuilder();
        canon.Append("{\"model\":").Append(JsonSerializer.Serialize(Llm14.Model));
        canon.Append(",\"prompt\":").Append(JsonSerializer.Serialize(prompt));
        canon.Append(",\"schema\":").Append(AlphaSchema14.Json);
        canon.Append(",\"temperature\":0.5");
        canon.Append(",\"url\":").Append(JsonSerializer.Serialize(Llm14.BaseUrl));
        canon.Append('}');
        byte[] hash = SHA256.HashData(Encoding.UTF8.GetBytes(canon.ToString()));
        return Convert.ToHexString(hash).ToLowerInvariant();
    }

    public static AlphaResponse14 RequestCached(int maxDayExcl, int seed, int callIndex,
                                                List<(double Value, string Pretty)> champions)
    {
        string prompt = Prompt(seed, callIndex, champions);
        string key = CacheKey(prompt);
        string path = Path.Combine(Llm14.CacheDir, key + ".json");
        if (File.Exists(path))
        {
            using var cached = JsonDocument.Parse(File.ReadAllText(path));
            var root = cached.RootElement;
            return new AlphaResponse14(
                root.GetProperty("content").GetString() ?? "",
                root.GetProperty("finish_reason").GetString() ?? "",
                root.TryGetProperty("usage", out var u) && u.TryGetProperty("completion_tokens", out var ct)
                    ? ct.GetInt32() : 0,
                FromCache: true);
        }
        string apiKey = Llm14.ApiKey();
        var request = new StringBuilder();
        request.Append("{\"model\":").Append(JsonSerializer.Serialize(Llm14.Model));
        request.Append(",\"messages\":[{\"role\":\"user\",\"content\":")
               .Append(JsonSerializer.Serialize(prompt)).Append("}]");
        request.Append(",\"temperature\":").Append(Llm14.Temperature.ToString(CultureInfo.InvariantCulture));
        request.Append(",\"max_tokens\":").Append(Llm14.MaxTokens);
        request.Append(",\"chat_template_kwargs\":{\"enable_thinking\":false}");
        request.Append(",\"response_format\":{\"type\":\"json_schema\",\"json_schema\":{\"name\":\"alpha_candidates\",\"strict\":true,\"schema\":")
               .Append(AlphaSchema14.Json).Append("}}");
        request.Append('}');
        string body = request.ToString();
        Exception last = null;
        for (int attempt = 0; attempt <= Llm14.MaxRetries; attempt++)
        {
            if (attempt > 0) Thread.Sleep(TimeSpan.FromSeconds(attempt));   // backoff 1 s puis 2 s
            try
            {
                using var http = new HttpRequestMessage(HttpMethod.Post, Llm14.BaseUrl + "/chat/completions");
                http.Headers.Authorization = new System.Net.Http.Headers.AuthenticationHeaderValue("Bearer", apiKey);
                http.Content = new StringContent(body, Encoding.UTF8, "application/json");
                using var sent = Llm14.Http.Send(http);
                sent.EnsureSuccessStatusCode();
                using var doc = JsonDocument.Parse(
                    sent.Content.ReadAsStringAsync().ConfigureAwait(false).GetAwaiter().GetResult());
                var choice = doc.RootElement.GetProperty("choices")[0];
                string content = choice.GetProperty("message").GetProperty("content").GetString() ?? "";
                string finish = choice.TryGetProperty("finish_reason", out var fr)
                    && fr.ValueKind == JsonValueKind.String ? fr.GetString() : "unknown";
                int compTokens = 0;
                if (doc.RootElement.TryGetProperty("usage", out var usage)
                    && usage.TryGetProperty("completion_tokens", out var tok)
                    && tok.ValueKind == JsonValueKind.Number) compTokens = tok.GetInt32();
                var payload = new StringBuilder();
                payload.Append("{\"content\":").Append(JsonSerializer.Serialize(content));
                payload.Append(",\"finish_reason\":").Append(JsonSerializer.Serialize(finish));
                payload.Append(",\"usage\":{\"completion_tokens\":").Append(compTokens).Append('}');
                payload.Append(",\"request\":{\"train_start\":\"")
                       .Append(Alpha13.Dates[Alpha13.FirstD].ToString("yyyy-MM-dd", CultureInfo.InvariantCulture));
                payload.Append("\",\"train_end\":\"")
                       .Append(Alpha13.Dates[Alpha13.FirstD + maxDayExcl - 1].ToString("yyyy-MM-dd", CultureInfo.InvariantCulture));
                payload.Append("\",\"seed\":").Append(seed).Append(",\"call\":").Append(callIndex).Append("}}");
                File.WriteAllText(path, payload.ToString(), new UTF8Encoding(false));
                return new AlphaResponse14(content, finish, compTokens, FromCache: false);
            }
            catch (Exception ex) when (ex is HttpRequestException or TaskCanceledException or TimeoutException or JsonException)
            { last = ex; }
        }
        throw new InvalidOperationException(
            "appel LLM echoue apres " + (Llm14.MaxRetries + 1) + " tentatives", last);
    }

    public sealed record LlmChampionResult14(GpNode Best, Dictionary<string, long> Stats,
                                             List<(double Value, string Pretty)> Champions);

    public static LlmChampionResult14 Champion(int maxDayExcl, int seed, long budget)
    {
        long start = FitnessCounter14.N;
        var stats = new Dictionary<string, long>();
        void Bump(string k) { stats.TryGetValue(k, out long v); stats[k] = v + 1; }
        long Get(string k) { stats.TryGetValue(k, out long v); return v; }
        var seen = new HashSet<string>();
        var champions = new List<(double Value, GpNode Node)>();
        var deadline = DateTime.UtcNow.AddSeconds(Llm14.WallClockSeconds);
        int callIndex = 0;
        while (FitnessCounter14.N - start < budget)
        {
            if (DateTime.UtcNow > deadline)
                throw new TimeoutException("budget horaire epuise avant le budget d'evaluations");
            long remaining = budget - (FitnessCounter14.N - start);
            int calls = (int)Math.Max(1, Math.Min(Llm14.Workers, remaining / Llm14.ExpectedPerCall));
            var prompts = champions.Select(c => (c.Value, Gp13.Pretty(c.Node))).ToList();
            var tasks = new Task<AlphaResponse14>[calls];
            for (int offset = 0; offset < calls; offset++)
            {
                int oi = offset;
                tasks[oi] = Task.Run(() => RequestCached(maxDayExcl, seed, callIndex + oi, prompts));
            }
            callIndex += calls;
            var responses = Task.WhenAll(tasks).ConfigureAwait(false).GetAwaiter().GetResult();
            foreach (var resp in responses)
            {
                Bump("calls");
                if (resp.FromCache) Bump("cache_hits");
                Bump("finish_" + (string.IsNullOrEmpty(resp.FinishReason) ? "none" : resp.FinishReason));
                stats["completion_tokens"] = Get("completion_tokens") + resp.CompletionTokens;
                if (!AlphaSchema14.Validate(resp.Content, out var candidates))
                { Bump("rejected_json"); continue; }
                foreach (var item in candidates)
                {
                    Bump("proposed");
                    try
                    {
                        if (item.ValueKind != JsonValueKind.Object) throw new FormatException("schema candidat");
                        var prefixProp = item.GetProperty("prefix");
                        if (prefixProp.ValueKind != JsonValueKind.String) throw new FormatException("schema candidat");
                        var node = AlphaParse14.Parse(prefixProp.GetString());
                        if (Gp13.Count(node) > (1 << (Gp13.MAX_DEPTH + 1)) - 1
                            || AlphaParse14.Depth(node) > Gp13.MAX_DEPTH)
                            throw new FormatException("arbre trop grand");
                        string key = Gp13.Pretty(node);
                        if (!seen.Add(key)) { Bump("duplicate"); continue; }
                        if (FitnessCounter14.N - start >= budget) { Bump("over_budget_proposed"); continue; }
                        double value = FitnessCounter14.On(node, maxDayExcl);
                        champions.Add((value, node));
                    }
                    catch (FormatException) { Bump("rejected_grammar"); }
                }
                champions.Sort((a, b) => b.Value.CompareTo(a.Value));
                if (champions.Count > 5) champions.RemoveRange(5, champions.Count - 5);
            }
        }
        long evaluated = FitnessCounter14.N - start;
        if (evaluated != budget)
            throw new InvalidOperationException($"budget LLM non tenu : {evaluated}/{budget} evaluations");
        if (champions.Count == 0)
            throw new InvalidOperationException("aucun candidat evalue sous le budget");
        stats["evaluated"] = evaluated;
        stats["accepted"] = Get("proposed") - Get("rejected_grammar") - Get("rejected_json") - Get("duplicate");
        return new LlmChampionResult14(
            champions[0].Node, stats,
            champions.Select(c => (c.Value, Gp13.Pretty(c.Node))).ToList());
    }
}

// Controles d'installation : le compteur compte, le prompt embarque grammaire et contexte.
FitnessCounter14.N = 0;
var probe14 = FitnessCounter14.On(Gp13.RandomTree(new Random(1), 2), 60);
if (FitnessCounter14.N != 1)
    throw new InvalidOperationException("le compteur de fitness n'a pas compte un unique appel");
FitnessCounter14.N = 0;
var probePrompt14 = AlphaLlm14.Prompt(0, 1, new List<(double, string)>());
if (!probePrompt14.Contains("Aucun candidat evalue.") || !probePrompt14.Contains("Appel 1, graine 0")
    || !probePrompt14.Contains("add(a,b), sub(a,b), mul(a,b), div(a,b), neg(a), abs(a)"))
    throw new InvalidOperationException("le prompt n'embarque pas grammaire et contexte attendus");

var sb15 = new System.Text.StringBuilder();
sb15.AppendLine($"Compteur de fitness installe : {FitnessCounter14.N} appel(s) avant la section 14");
sb15.AppendLine($"Boucle LLM prete : {Llm14.Batch} propositions par appel, {Llm14.Workers} appels simultanes, garde horaire {Llm14.WallClockSeconds:F0} s");
sb15.ToString().Display();

Compteur de fitness installe : 0 appel(s) avant la section 14
Boucle LLM prete : 40 propositions par appel, 6 appels simultanes, garde horaire 2400 s


### La grille : deux bras, une seule porte comptée

La cellule suivante exécute la comparaison complète : 5 graines × 5 plis walk-forward (les mêmes bornes que la section 13). Pour chaque pli : le compteur est remis à zéro, le bras AG évolue — et dépense un budget **mesuré** — puis le bras LLM reçoit exactement ce budget. Les deux champions sont ensuite évalués hors échantillon sur le pli : PnL quotidien, Sharpe, MSE de la prévision (score standardisé contre rendement futur standardisé, en coupe par jour) et biais signé.

Un décompte à ne pas confondre : « évaluations » ne veut pas dire « appels réseau ». Un appel au service propose jusqu'à 40 candidats ; ceux qui passent la grammaire et n'ont pas déjà été vus coûtent chacun **une** évaluation ; les doublons et les rejets sont gratuits. C'est cette définition qui rend la comparaison honnête : le LLM n'est pas pénalisé pour proposer des formes déjà vues — il ne les paie tout simplement pas.

### Grille walk-forward à budget égal

Pour chacune des 25 combinaisons (5 plis × 5 graines) : l'AG évolue d'abord sur la fenêtre d'entraînement, le compteur relève ce qu'elle a **réellement** consommé, puis le bras LLM dispose d'exactement ce nombre d'évaluations. Les deux champions tradent ensuite le même pli, hors échantillon, dans le même moteur de portefeuille.

La perte de précision qui sert au test de Diebold-Mariano est définie explicitement : à chaque date, le score du champion et le rendement futur sont **standardisés en coupe** (moyenne et écart-type de la date), puis comparés par erreur quadratique. Une même transformation est appliquée aux deux bras, et elle est insensible à l'échelle arbitraire des scores — ce que le Sharpe, lui, ne mesure pas.

In [22]:
// --- Grille walk-forward : AG et LLM, budget d'evaluations mesure identique ---
using System.Diagnostics;

int[] SEEDS_LLM = { 0, 1, 7, 42, 99 };
const int N_FOLDS_LLM = 5;
int oosStart14 = Alpha13.NDays - 504, oosEnd14 = Alpha13.NDays;   // exactement les plis de la section 13
int foldLen14 = (oosEnd14 - oosStart14 + N_FOLDS_LLM - 1) / N_FOLDS_LLM;

// Perte de precision : score standardise en coupe contre rendement futur standardise.
// Retourne la MSE par date (perte du test DM) et le biais signe moyen.
(double[] MsePerDay, double BiasMean) Quality14(GpNode tree, int fromDay, int toDayExcl)
{
    var scores = Gp13.EvalAll(tree);
    int nT = Alpha13.NTicks;
    var mses = new List<double>();
    double sum = 0; int cnt = 0;
    for (int day = fromDay; day < toDayExcl; day++)
    {
        int o0 = day * nT;
        double ms = 0, mr = 0;
        for (int t = 0; t < nT; t++) { ms += scores[o0 + t]; mr += Alpha13.RetNext[o0 + t]; }
        ms /= nT; mr /= nT;
        double ss = 0, sr = 0;
        for (int t = 0; t < nT; t++)
        {
            double es = scores[o0 + t] - ms, er = Alpha13.RetNext[o0 + t] - mr;
            ss += es * es; sr += er * er;
        }
        double sdS = Math.Sqrt(ss / (nT - 1)), sdR = Math.Sqrt(sr / (nT - 1));
        double mseDay = 0;
        for (int t = 0; t < nT; t++)
        {
            // score plat (expression constante) : idiome C# de la section 13, position neutre en coupe
            double zs = sdS > 1e-9 ? (scores[o0 + t] - ms) / sdS : 0.0;
            double zr = sdR > 1e-9 ? (Alpha13.RetNext[o0 + t] - mr) / sdR : 0.0;
            double d = zs - zr;
            mseDay += d * d; sum += d; cnt++;
        }
        mses.Add(mseDay / nT);
    }
    return (mses.ToArray(), cnt > 0 ? sum / cnt : 0.0);
}

var swAll14 = Stopwatch.StartNew();
var gridRows14 = new List<(int Seed, int FoldStart, long Budget, double SharpeGa, double SharpeLlm, string Ga, string Llm, long Calls)>();
var pnlGaBySeed = new Dictionary<int, List<double>>();
var pnlLlmBySeed = new Dictionary<int, List<double>>();
var mseGaBySeed = new Dictionary<int, List<double>>();
var mseLlmBySeed = new Dictionary<int, List<double>>();
var biasBySeed14 = new Dictionary<int, (double Ga, double Llm)>();

var sbGrid = new System.Text.StringBuilder();
foreach (int seed in SEEDS_LLM)
{
    var gaPnl = new List<double>(); var llmPnl = new List<double>();
    var gaMse = new List<double>(); var llmMse = new List<double>();
    var gaBias = new List<double>(); var llmBias = new List<double>();
    for (int k = 0; k < N_FOLDS_LLM; k++)
    {
        int fs = oosStart14 + k * foldLen14;
        int fe = Math.Min(fs + foldLen14, oosEnd14);

        FitnessCounter14.N = 0;                                   // bras AG compte, lui aussi
        var champGa = AlphaLlm14.EvolveChampionCounted(fs, seed);
        long budget = FitnessCounter14.N;
        var pnlGa = Engine13.DailyPnl(Engine13.Weights(Gp13.EvalAll(champGa)));
        var (mseGa, biasGa) = Quality14(champGa, fs, fe);

        var swFold = Stopwatch.StartNew();
        var res = AlphaLlm14.Champion(fs, seed, budget);
        swFold.Stop();
        var champLlm = res.Best;
        var pnlLlm = Engine13.DailyPnl(Engine13.Weights(Gp13.EvalAll(champLlm)));
        var (mseLlm, biasLlm) = Quality14(champLlm, fs, fe);

        for (int day = fs; day < fe; day++) { gaPnl.Add(pnlGa[day]); llmPnl.Add(pnlLlm[day]); }
        gaMse.AddRange(mseGa); llmMse.AddRange(mseLlm);
        gaBias.Add(biasGa); llmBias.Add(biasLlm);
        gridRows14.Add((seed, fs, budget, Engine13.Sharpe(pnlGa, fs, fe), Engine13.Sharpe(pnlLlm, fs, fe),
                        Gp13.Pretty(champGa), Gp13.Pretty(champLlm), res.Stats.TryGetValue("calls", out var c) ? c : 0));
        sbGrid.AppendLine($"graine {seed,2}  pli jour {fs,4}  budget {budget,3}  Sharpe AG {Engine13.Sharpe(pnlGa, fs, fe),6:F3}  LLM {Engine13.Sharpe(pnlLlm, fs, fe),6:F3}  {(res.Stats.TryGetValue("calls", out var c2) ? c2 : 0),2} appels  {swFold.Elapsed.TotalSeconds,4:F0} s");
        sbGrid.AppendLine($"    AG  : {Gp13.Pretty(champGa).Substring(0, Math.Min(96, Gp13.Pretty(champGa).Length))}");
        sbGrid.AppendLine($"    LLM : {Gp13.Pretty(champLlm).Substring(0, Math.Min(96, Gp13.Pretty(champLlm).Length))}");
        sbGrid.ToString().Display();
        sbGrid.Clear();
    }
    pnlGaBySeed[seed] = gaPnl; pnlLlmBySeed[seed] = llmPnl;
    mseGaBySeed[seed] = gaMse; mseLlmBySeed[seed] = llmMse;
    biasBySeed14[seed] = (gaBias.Average(), llmBias.Average());
}
swAll14.Stop();

var budgets14 = gridRows14.Select(r => (double)r.Budget).OrderBy(v => v).ToArray();
double Mean14(IEnumerable<double> xs) { double s = 0; int n = 0; foreach (var x in xs) { s += x; n++; } return n > 0 ? s / n : 0; }
double Sd14(IEnumerable<double> xs)
{
    var a = xs.ToArray(); if (a.Length < 2) return 0;
    double m = Mean14(a), s = 0; foreach (var x in a) { double e = x - m; s += e * e; }
    return Math.Sqrt(s / (a.Length - 1));
}
var sb16 = new System.Text.StringBuilder();
sb16.AppendLine($"{N_FOLDS_LLM} plis x {SEEDS_LLM.Length} graines ; budget d'evaluations identique dans les deux bras ; duree {swAll14.Elapsed.TotalSeconds:F0} s");
sb16.AppendLine($"Budget par pli : min {budgets14[0]:F0}, mediane {budgets14[budgets14.Length / 2]:F0}, max {budgets14[^1]:F0} evaluations");
sb16.ToString().Display();

graine  0  pli jour 1965  budget 1050  Sharpe AG -0,440  LLM -0,386  135 appels     7 s
    AG  : abs(add(abs(mom), volr))
    LLM : abs(mul(mom, sub(mom, mul(volr, 2))))


graine  0  pli jour 2066  budget 1050  Sharpe AG  1,557  LLM  2,685  79 appels     6 s
    AG  : abs(mom)
    LLM : add(abs(mom), abs(sub(volr, mom)))


graine  0  pli jour 2167  budget 1050  Sharpe AG  0,379  LLM -0,320  178 appels     6 s
    AG  : abs(add(r21, abs(sub(mom, r5))))
    LLM : mul(abs(mom), abs(sub(mom, mul(volr, 2))))


graine  0  pli jour 2268  budget 1050  Sharpe AG  1,410  LLM  0,933  167 appels     6 s
    AG  : abs(add(r21, abs(sub(mom, r5))))
    LLM : add(abs(mom), abs(sub(mom, volr)))


graine  0  pli jour 2369  budget 1050  Sharpe AG -0,067  LLM -0,808  133 appels     6 s
    AG  : abs(add(r21, add(r21, abs(mom))))
    LLM : add(abs(mom), abs(volr))


graine  1  pli jour 1965  budget 1050  Sharpe AG -0,725  LLM -0,170  76 appels     7 s
    AG  : add(r21, div(add(volr, sub(volr, r21)), div(neg(1.2), add(r1, r1))))
    LLM : add(abs(mom), abs(sub(mom, volr)))


... (16 outputs left out)

graine 99  pli jour 2167  budget 1050  Sharpe AG  0,323  LLM  0,195  104 appels  1889 s
    AG  : add(abs(mom), abs(add(volr, r5)))
    LLM : add(abs(mom), abs(sub(mom, volr)))


graine 99  pli jour 2268  budget 1050  Sharpe AG  1,156  LLM  0,716  95 appels  1055 s
    AG  : abs(add(r21, add(r21, abs(mom))))
    LLM : add(abs(mom), abs(sub(abs(r21), volr)))


graine 99  pli jour 2369  budget 1050  Sharpe AG -0,311  LLM -0,808  91 appels  1089 s
    AG  : add(abs(mom), r21)
    LLM : add(abs(mom), abs(volr))


5 plis x 5 graines ; budget d'evaluations identique dans les deux bras ; duree 7628 s
Budget par pli : min 1050, mediane 1050, max 1050 evaluations


### Verdict : le bras LLM bat-il l'AG, à budget égal ?

Le critère est celui de la doctrine §C du dépôt, appliqué ici à la comparaison **LLM contre AG** : edge quotidien moyen inter-graines ≥ 2σ inter-graines **et** Diebold-Mariano médian `p < 0,05` sur la perte de précision. Un edge seul ne suffit pas (le σ peut être large), un `p` seul non plus (il ne dit rien du signe).

In [23]:
// --- Verdict : LLM contre AG a budget egal, puis positionnement contre les facteurs fixes ---
var edgesLlmGa = new Dictionary<int, double>();
var dmLlmGa = new Dictionary<int, double>();
foreach (int seed in SEEDS_LLM)
{
    var ga = pnlGaBySeed[seed]; var llm = pnlLlmBySeed[seed];
    double edge = 0;
    for (int i = 0; i < ga.Count; i++) edge += llm[i] - ga[i];
    edgesLlmGa[seed] = edge / ga.Count * 10000.0;
    dmLlmGa[seed] = DmPvalue(mseLlmBySeed[seed].ToArray(), mseGaBySeed[seed].ToArray());
}

double edgeMean14 = Mean14(edgesLlmGa.Values);
double edgeSe14 = Sd14(edgesLlmGa.Values) / Math.Sqrt(edgesLlmGa.Count);
double dmMedian14 = dmLlmGa.Values.OrderBy(v => v).ElementAt(dmLlmGa.Count / 2);
string verdict14 = "INCONCLUSIVE";
if (edgeMean14 >= 2 * edgeSe14 && dmMedian14 < 0.05) verdict14 = "BEATS";
else if (edgeMean14 <= -2 * edgeSe14 && dmMedian14 < 0.05) verdict14 = "NO BEATS (dominee)";

var sb17 = new System.Text.StringBuilder();
sb17.AppendLine("Perte de precision (MSE en coupe), par graine :");
foreach (int seed in SEEDS_LLM)
    sb17.AppendLine($"  graine {seed,2}  AG {Mean14(mseGaBySeed[seed]):F4}  LLM {Mean14(mseLlmBySeed[seed]):F4}  edge LLM-AG {edgesLlmGa[seed],8:+0.00;-0.00} bps/j  DM p {dmLlmGa[seed]:F4}");
sb17.AppendLine();
sb17.AppendLine($"Edge moyen LLM - AG : {edgeMean14:+0.00;-0.00} bps/jour (SE inter-graines {edgeSe14:F2} ; seuil 2-sigma = +/-{2 * edgeSe14:F2})");
sb17.AppendLine($"DM p-valeur mediane ({SEEDS_LLM.Length} graines) : {dmMedian14:F4}");

var sharpeGa14 = SEEDS_LLM.Select(s => Engine13.Sharpe(pnlGaBySeed[s].ToArray())).ToArray();
var sharpeLlm14 = SEEDS_LLM.Select(s => Engine13.Sharpe(pnlLlmBySeed[s].ToArray())).ToArray();
sb17.AppendLine();
sb17.AppendLine($"Sharpe hors echantillon, moyenne inter-graines : AG {Mean14(sharpeGa14):F3} (sd {Sd14(sharpeGa14):F3}) ; LLM {Mean14(sharpeLlm14):F3} (sd {Sd14(sharpeLlm14):F3})");

double biasGa14 = Mean14(biasBySeed14.Values.Select(b => b.Ga));
double biasLlm14 = Mean14(biasBySeed14.Values.Select(b => b.Llm));
sb17.AppendLine($"Biais signe moyen (score standardise - rendement standardise) : AG {biasGa14:+0.0000;-0.0000} ; LLM {biasLlm14:+0.0000;-0.0000}");

// Baseline majoritaire : prevoir un rendement net nul a chaque date = rester sans position.
// Les poids dollar-neutres la neutralisent par construction, donc l'edge se lit
// directement sur le PnL moyen de chaque bras (frais deja deduits, 5 bps par cote).
double meanGa14 = Mean14(pnlGaBySeed.Values.Select(p => Mean14(p)));
double meanLlm14 = Mean14(pnlLlmBySeed.Values.Select(p => Mean14(p)));
sb17.AppendLine();
sb17.AppendLine($"Baseline majoritaire (prevision nulle = sans position) : edge AG {meanGa14 * 10000.0:+0.00;-0.00} bps/j ; LLM {meanLlm14 * 10000.0:+0.00;-0.00} bps/j");

// Facteurs fixes sur la meme fenetre hors echantillon (graine 0, comme cote Python).
int fs0 = oosStart14, fe0 = oosEnd14;
var fixedFactors14 = new (string Name, double[] Pnl)[]
{
    ("momentum 12-1", pnlMom),
    ("vol-inverse", pnlVolInv),
    ("panier equipondere", pnlEq),
};
sb17.AppendLine();
sb17.AppendLine("Facteurs fixes sur la meme fenetre hors echantillon :");
foreach (var (name, series) in fixedFactors14)
{
    var pnlFixed = new double[fe0 - fs0];
    for (int day = fs0; day < fe0; day++) pnlFixed[day - fs0] = series[day];
    var llm0 = pnlLlmBySeed[SEEDS_LLM[0]].ToArray();
    var ga0 = pnlGaBySeed[SEEDS_LLM[0]].ToArray();
    double edgeLlm = 0, edgeGa = 0;
    for (int i = 0; i < pnlFixed.Length; i++)
    {
        edgeLlm += llm0[i] - pnlFixed[i];
        edgeGa += ga0[i] - pnlFixed[i];
    }
    sb17.AppendLine($"  {name,-22} Sharpe {Engine13.Sharpe(pnlFixed),6:F3}  edge LLM {edgeLlm / pnlFixed.Length * 10000.0,8:+0.00;-0.00} bps/j  edge AG {edgeGa / pnlFixed.Length * 10000.0,8:+0.00;-0.00} bps/j");
}

sb17.AppendLine();
sb17.AppendLine($"VERDICT (LLM contre AG, budget d'evaluations egal) : {verdict14}");
sb17.ToString().Display();

Perte de precision (MSE en coupe), par graine :
  graine  0  AG 1,7112  LLM 1,7322  edge LLM-AG    -1,65 bps/j  DM p 0,3988
  graine  1  AG 1,7348  LLM 1,7158  edge LLM-AG    +3,03 bps/j  DM p 0,5635
  graine  7  AG 1,2251  LLM 1,7439  edge LLM-AG    +2,61 bps/j  DM p 0,0000
  graine 42  AG 1,7362  LLM 1,7359  edge LLM-AG    -0,16 bps/j  DM p 0,9804
  graine 99  AG 1,7355  LLM 1,7366  edge LLM-AG    -1,26 bps/j  DM p 0,9690

Edge moyen LLM - AG : +0,52 bps/jour (SE inter-graines 0,97 ; seuil 2-sigma = +/-1,95)
DM p-valeur mediane (5 graines) : 0,5635

Sharpe hors echantillon, moyenne inter-graines : AG -0,026 (sd 0,426) ; LLM 0,093 (sd 0,215)
Biais signe moyen (score standardise - rendement standardise) : AG -+0,0000 ; LLM -+0,0000

Baseline majoritaire (prevision nulle = sans position) : edge AG +0,03 bps/j ; LLM +0,54 bps/j

Facteurs fixes sur la meme fenetre hors echantillon :
  momentum 12-1          Sharpe  0,187  edge LLM    +0,22 bps/j  edge AG    +1,87 bps/j
  vol-inverse      

### Lecture du verdict

Le verdict est **INCONCLUSIVE** : à budget d'évaluations mesuré et imposé (1 050 par pli, la même porte de fitness pour les deux bras), le bras LLM ne se distingue pas de l'AG.

**Ce que dit l'organe.** Edge moyen hors-échantillon LLM − AG de **+0,52 bps/jour**, soit **0,54 σ** inter-graines (seuil 2σ = ±1,95) ; Diebold-Mariano médian **0,5635** (seuil 0,05). Aucune des deux conjonctions de la doctrine §C n'est atteinte — dans un sens comme dans l'autre. Les Sharpe hors-échantillon moyens (AG −0,026, LLM +0,093) restent tous deux proches de zéro : les deux bras produisent du bruit, pas un signal.

**La seule graine « significative » est une contradiction entre les deux jambes du test, pas un alpha LLM.** Sur la graine 7, DM tombe à 0,0000 avec un edge de +2,61 bps/j. La lecture du champion explique le p : l'AG y sélectionne `div(r5, r5)` — une **constante** — sur 3 plis sur 5. Avec `Quality14`, un score sans dispersion reçoit `zs = 0` (garde `sdS > 1e-9`), donc une prévision constante a une perte égale à la moyenne de `zr²`, soit (n−1)/n = 7/8 = **0,875** pour 8 titres — par construction, quels que soient les rendements. Les champions réels des deux autres plis perdent ≈ **1,73**, environ deux fois la prévision nulle : une corrélation en coupe à peu près nulle avec le rendement. La moyenne de la graine, (3 × 0,875 + 2 × 1,73)/5 ≈ **1,2251**, retrouve la sortie. À p = 0,0000, le DM dit donc que **l'AG est plus précis** que le LLM — la prévision plate est structurellement avantagée par cette perte — tandis que l'edge favorise le LLM (+2,61 bps/j) : les deux jambes de la conjonction §C se contredisent sur cette graine, ce qui exclut le verdict à lui seul. Ce n'est pas la variance du perdant qui a rendu l'écart significatif — c'est une prévision plate récompensée par construction, la même trivialité que le panier équipondéré du paragraphe suivant récompense à l'échelle du portefeuille.

**Le contexte rend la leçon plus nette encore.** Sur la même fenêtre hors-échantillon, le **panier équipondéré** poste un Sharpe de **1,594**, devant tous les candidats des deux bras (edges de −7,42 et −5,77 bps/j face à lui) ; le momentum 12-1 fixe (Sharpe 0,187) fait mieux que les deux bras contre la baseline, sans les approcher. Ne rien sélectionner et tout détenir bat ici les deux producteurs de candidats — le même mode d'échec que le jumeau Python (section 14 de `Search-05-GeneticAlgorithms.ipynb`, verdict INCONCLUSIVE lui aussi, avec un signe d'edge opposé : −0,41 bps/j en faveur de l'AG côté Python contre +0,52 en faveur du LLM ici — les deux sous leur seuil 2σ, leur signe n'est pas un résultat), obtenu ici sous la garantie du compteur unique : budget mesuré, jamais supposé.

**Ce que la section établit malgré tout.** Le service local a émis des expressions dans la grammaire bornée, évaluées par la même fitness que celle de l'AG, jusqu'à épuisement exact du budget mesuré de l'AG. Le protocole est le livrable : il permet de dire « pas de différence » avec une exigence symétrique sur les deux bras, plutôt que de comparer des tours de calendrier différents.

***

## Références et provenance

- Port C# : EPIC [#4956](https://github.com/jsboige/CoursIA/issues/4956), prong A [#3801](https://github.com/jsboige/CoursIA/issues/3801).
- Tables texte : verdict #3436 INTRINSIC.
